In [ ]:
# @title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title AI Flows HUB (Warm-UP Tools)
# ============================================================
# AI FLOWS HUB — ALL 3 TOOLS WARM-UP
# Kaggle + Colab
#
# ONE CELL:
#   ✓ Whisper large-v3-turbo
#   ✓ Real-ESRGAN x4 / x2
#   ✓ Chatterbox Turbo
#
# This combines the existing working warm-ups without changing
# their core behavior. Each tool is prepared independently.
#
# IMPORTANT:
#   - Models are cached/downloaded.
#   - GPU memory is released after warm-up.
#   - Runners are created exactly as in the working versions.
# ============================================================

print("\n" + "=" * 78)
print("🔥 ALL 3 TOOLS — WARM-UP START")
print("=" * 78)

# ============================================================
# 1/3 — WHISPER
# ============================================================

print("\n" + "=" * 78)
print("🎙️ 1/3 — WHISPER WARM-UP")
print("=" * 78)

# @title
# ============================================================
# WHISPER WARM-UP — COLAB / KAGGLE
# Installs dependencies + caches large-v3-turbo
# ============================================================

!pip install -q -U openai-whisper
!apt-get update -qq && apt-get install -y -qq ffmpeg

import whisper
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "large-v3-turbo"

print(f"🖥️ Device: {DEVICE.upper()}")

if DEVICE == "cpu":
    print("⚠️ WARNING: GPU not detected.")

print(f"⬇️ Caching Whisper {MODEL_NAME}...")

# Load once only to download/cache the model.
model = whisper.load_model(MODEL_NAME, device=DEVICE)

print("✅ Whisper warm-up complete!")
print(f"✅ Model: {MODEL_NAME}")

# Important: warm-up should NOT leave the model in GPU memory.
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("🧹 GPU memory released.")

# ============================================================
# 2/3 — REAL-ESRGAN
# ============================================================

print("\n" + "=" * 78)
print("🖼️ 2/3 — REAL-ESRGAN WARM-UP")
print("=" * 78)

# ============================================================
# REAL-ESRGAN WARM-UP — KAGGLE + COLAB
#
# Does:
#   ✓ Detects Kaggle / Colab
#   ✓ Checks existing dependencies
#   ✓ Downloads model weights
#   ✓ Creates standalone runner
#
# Does NOT:
#   ✗ Reinstall NumPy
#   ✗ Reinstall Pillow
#   ✗ Load model into GPU
# ============================================================

import os
import sys
import urllib.request
from pathlib import Path

# ============================================================
# PLATFORM
# ============================================================

IS_KAGGLE = os.path.exists("/kaggle/working")

BASE_DIR = (
    "/kaggle/working"
    if IS_KAGGLE
    else "/content"
)

MODEL_DIR = os.path.join(
    BASE_DIR,
    "realesrgan_models"
)

RUNNER_PATH = os.path.join(
    BASE_DIR,
    "run_realesrgan.py"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

print("=" * 70)
print("🚀 REAL-ESRGAN WARM-UP")
print("=" * 70)

print(
    f"Platform : "
    f"{'Kaggle' if IS_KAGGLE else 'Colab'}"
)

print(
    f"Python   : "
    f"{sys.version.split()[0]}"
)

print(
    f"Base dir : "
    f"{BASE_DIR}"
)

# ============================================================
# CHECK EXISTING DEPENDENCIES
# ============================================================

print("\n🔍 Checking existing dependencies...")

try:

    import torch
    import numpy
    from PIL import Image

    print(
        f"✅ PyTorch : {torch.__version__}"
    )

    print(
        f"✅ NumPy   : {numpy.__version__}"
    )

    print(
        "✅ Pillow  : ready"
    )

    print(
        f"✅ CUDA    : "
        f"{torch.cuda.is_available()}"
    )

except Exception as e:

    raise RuntimeError(
        "Required runtime packages are missing: "
        f"{e}"
    )

# ============================================================
# MODEL WEIGHTS
# ============================================================

MODELS = {
    "RealESRGAN_x4plus.pth":
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",

    "RealESRGAN_x2plus.pth":
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
}

print("\n⬇️ Checking Real-ESRGAN models...")

for name, url in MODELS.items():

    path = os.path.join(
        MODEL_DIR,
        name
    )

    if os.path.isfile(path):

        size_mb = (
            os.path.getsize(path)
            / 1024
            / 1024
        )

        print(
            f"✅ {name} "
            f"({size_mb:.1f} MB)"
        )

        continue

    print(
        f"⬇️ Downloading {name}..."
    )

    urllib.request.urlretrieve(
        url,
        path
    )

    size_mb = (
        os.path.getsize(path)
        / 1024
        / 1024
    )

    print(
        f"✅ Downloaded "
        f"{name} "
        f"({size_mb:.1f} MB)"
    )

# ============================================================
# CREATE RUNNER
#
# IMPORTANT:
# Keep the EXACT architecture from the working version.
# ============================================================

RUNNER_CODE = r'''
import os
import sys
import math
import time
import argparse
import tempfile
import shutil
import zipfile

from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F


IMG_EXTS = (
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".webp",
    ".tiff",
    ".tif",
)


def log(message=""):
    print(message, flush=True)


def format_seconds(seconds):
    seconds = max(0.0, float(seconds))

    if seconds < 60:
        return f"{seconds:.1f}s"

    minutes = int(seconds // 60)
    secs = int(seconds % 60)

    if minutes < 60:
        return f"{minutes}m {secs}s"

    hours = minutes // 60
    minutes %= 60

    return f"{hours}h {minutes}m {secs}s"


def print_gpu_memory():

    if not torch.cuda.is_available():
        return

    allocated = (
        torch.cuda.memory_allocated()
        / (1024 ** 3)
    )

    reserved = (
        torch.cuda.memory_reserved()
        / (1024 ** 3)
    )

    log(
        f"       GPU VRAM: "
        f"{allocated:.2f} GB allocated / "
        f"{reserved:.2f} GB reserved"
    )


# ============================================================
# EXACT REAL-ESRGAN ARCHITECTURE
# ============================================================

class ResidualDenseBlock(nn.Module):

    def __init__(
        self,
        num_feat=64,
        num_grow_ch=32,
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            num_feat,
            num_grow_ch,
            3,
            1,
            1,
        )

        self.conv2 = nn.Conv2d(
            num_feat + num_grow_ch,
            num_grow_ch,
            3,
            1,
            1,
        )

        self.conv3 = nn.Conv2d(
            num_feat + 2 * num_grow_ch,
            num_grow_ch,
            3,
            1,
            1,
        )

        self.conv4 = nn.Conv2d(
            num_feat + 3 * num_grow_ch,
            num_grow_ch,
            3,
            1,
            1,
        )

        self.conv5 = nn.Conv2d(
            num_feat + 4 * num_grow_ch,
            num_feat,
            3,
            1,
            1,
        )

        self.lrelu = nn.LeakyReLU(
            0.2,
            inplace=True,
        )

    def forward(self, x):

        x1 = self.lrelu(
            self.conv1(x)
        )

        x2 = self.lrelu(
            self.conv2(
                torch.cat(
                    [x, x1],
                    1,
                )
            )
        )

        x3 = self.lrelu(
            self.conv3(
                torch.cat(
                    [x, x1, x2],
                    1,
                )
            )
        )

        x4 = self.lrelu(
            self.conv4(
                torch.cat(
                    [x, x1, x2, x3],
                    1,
                )
            )
        )

        x5 = self.conv5(
            torch.cat(
                [x, x1, x2, x3, x4],
                1,
            )
        )

        return x5 * 0.2 + x


class RRDB(nn.Module):

    def __init__(
        self,
        num_feat,
        num_grow_ch=32,
    ):

        super().__init__()

        self.rdb1 = ResidualDenseBlock(
            num_feat,
            num_grow_ch,
        )

        self.rdb2 = ResidualDenseBlock(
            num_feat,
            num_grow_ch,
        )

        self.rdb3 = ResidualDenseBlock(
            num_feat,
            num_grow_ch,
        )

    def forward(self, x):

        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)

        return out * 0.2 + x


class RRDBNet(nn.Module):

    def __init__(
        self,
        num_in_ch=3,
        num_out_ch=3,
        num_feat=64,
        num_block=23,
        num_grow_ch=32,
        scale=4,
    ):

        super().__init__()

        self.scale = scale

        if scale == 2:

            self.pixel_unshuffle = (
                nn.PixelUnshuffle(2)
            )

            num_in_ch *= 4

        self.conv_first = nn.Conv2d(
            num_in_ch,
            num_feat,
            3,
            1,
            1,
        )

        self.body = nn.ModuleList(
            [
                RRDB(
                    num_feat,
                    num_grow_ch,
                )
                for _ in range(num_block)
            ]
        )

        self.conv_body = nn.Conv2d(
            num_feat,
            num_feat,
            3,
            1,
            1,
        )

        self.conv_up1 = nn.Conv2d(
            num_feat,
            num_feat,
            3,
            1,
            1,
        )

        self.conv_up2 = nn.Conv2d(
            num_feat,
            num_feat,
            3,
            1,
            1,
        )

        self.conv_hr = nn.Conv2d(
            num_feat,
            num_feat,
            3,
            1,
            1,
        )

        self.conv_last = nn.Conv2d(
            num_feat,
            num_out_ch,
            3,
            1,
            1,
        )

        self.lrelu = nn.LeakyReLU(
            0.2,
            inplace=True,
        )

    def forward(self, x):

        if self.scale == 2:
            x = self.pixel_unshuffle(x)

        feat = self.conv_first(x)

        body_feat = feat

        for block in self.body:
            body_feat = block(body_feat)

        body_feat = self.conv_body(
            body_feat
        )

        feat = feat + body_feat

        feat = self.lrelu(
            self.conv_up1(
                F.interpolate(
                    feat,
                    scale_factor=2,
                    mode="nearest",
                )
            )
        )

        feat = self.lrelu(
            self.conv_up2(
                F.interpolate(
                    feat,
                    scale_factor=2,
                    mode="nearest",
                )
            )
        )

        feat = self.lrelu(
            self.conv_hr(feat)
        )

        return self.conv_last(feat)


# ============================================================
# MODEL
# ============================================================

def load_model(
    model_path,
    device,
):

    name = os.path.basename(
        model_path
    ).lower()

    native_scale = (
        2
        if "x2" in name
        else 4
    )

    log(
        "[Real-ESRGAN] Creating model..."
    )

    model = RRDBNet(
        scale=native_scale,
        num_block=23,
    ).to(device)

    log(
        "[Real-ESRGAN] Loading weights..."
    )

    state_dict = torch.load(
        model_path,
        map_location=device,
        weights_only=False,
    )

    if "params_ema" in state_dict:
        state_dict = state_dict["params_ema"]

    state_dict = {
        k.replace("module.", ""): v
        for k, v in state_dict.items()
    }

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    model.eval()

    return model, native_scale


# ============================================================
# TILED UPSCALER
# ============================================================

def upscale_image(
    image_np,
    model,
    model_scale,
    device,
    tile_size=512,
    tile_pad=32,
):

    image = (
        image_np.astype(np.float32)
        / 255.0
    )

    tensor = torch.from_numpy(
        np.transpose(
            image,
            (2, 0, 1),
        )
    ).float().unsqueeze(0).to(device)

    scale = model_scale

    _, channels, height, width = (
        tensor.shape
    )

    mod_pad_h = (
        scale - height % scale
    ) % scale

    mod_pad_w = (
        scale - width % scale
    ) % scale

    if mod_pad_h or mod_pad_w:

        tensor = F.pad(
            tensor,
            (
                0,
                mod_pad_w,
                0,
                mod_pad_h,
            ),
            mode="reflect",
        )

    _, _, height, width = tensor.shape

    output = tensor.new_zeros(
        (
            1,
            channels,
            height * scale,
            width * scale,
        )
    )

    tiles_x = math.ceil(
        width / tile_size
    )

    tiles_y = math.ceil(
        height / tile_size
    )

    total_tiles = (
        tiles_x * tiles_y
    )

    log(
        f"       Tiles         : "
        f"{tiles_x} x {tiles_y} = "
        f"{total_tiles}"
    )

    log(
        f"       Tile progress : "
        f"0/{total_tiles}"
    )

    tile_number = 0

    for y in range(tiles_y):

        for x in range(tiles_x):

            tile_number += 1

            start_x = (
                x * tile_size
            )

            start_y = (
                y * tile_size
            )

            end_x = min(
                start_x + tile_size,
                width,
            )

            end_y = min(
                start_y + tile_size,
                height,
            )

            pad_start_x = max(
                start_x - tile_pad,
                0,
            )

            pad_start_y = max(
                start_y - tile_pad,
                0,
            )

            pad_end_x = min(
                end_x + tile_pad,
                width,
            )

            pad_end_y = min(
                end_y + tile_pad,
                height,
            )

            tile = tensor[
                :,
                :,
                pad_start_y:pad_end_y,
                pad_start_x:pad_end_x,
            ]

            tile_start = time.time()

            with torch.no_grad():

                output_tile = model(
                    tile
                )

            tile_time = (
                time.time()
                - tile_start
            )

            output_start_x = (
                start_x * scale
            )

            output_end_x = (
                end_x * scale
            )

            output_start_y = (
                start_y * scale
            )

            output_end_y = (
                end_y * scale
            )

            output_start_x_tile = (
                (start_x - pad_start_x)
                * scale
            )

            output_start_y_tile = (
                (start_y - pad_start_y)
                * scale
            )

            output_width = (
                (end_x - start_x)
                * scale
            )

            output_height = (
                (end_y - start_y)
                * scale
            )

            output[
                :,
                :,
                output_start_y:output_end_y,
                output_start_x:output_end_x,
            ] = output_tile[
                :,
                :,
                output_start_y_tile:
                output_start_y_tile + output_height,
                output_start_x_tile:
                output_start_x_tile + output_width,
            ]

            log(
                f"       Tile "
                f"{tile_number}/{total_tiles} "
                f"({tile_number / total_tiles * 100:.1f}%) "
                f"- {tile_time:.2f}s"
            )

            if (
                tile_number == 1
                or tile_number == total_tiles
                or tile_number % 5 == 0
            ):
                print_gpu_memory()

    if mod_pad_h:

        output = output[
            :,
            :,
            :-mod_pad_h * scale,
            :,
        ]

    if mod_pad_w:

        output = output[
            :,
            :,
            :,
            :-mod_pad_w * scale,
        ]

    output = (
        output
        .squeeze()
        .float()
        .cpu()
        .clamp_(0, 1)
        .numpy()
    )

    output = np.transpose(
        output,
        (1, 2, 0),
    )

    return (
        output * 255.0
    ).round().astype(np.uint8)


# ============================================================
# SINGLE IMAGE
# ============================================================

def process_single(
    input_path,
    output_path,
    model,
    model_scale,
    target_scale,
    device,
    tile_size,
    tile_pad,
):

    start = time.time()

    with Image.open(
        input_path
    ) as image:

        image = image.convert(
            "RGB"
        )

    image_np = np.array(
        image
    )

    height, width = (
        image_np.shape[:2]
    )

    log(
        f"       Resolution    : "
        f"{width}x{height}"
    )

    output_np = upscale_image(
        image_np,
        model,
        model_scale,
        device,
        tile_size,
        tile_pad,
    )

    if abs(
        target_scale - model_scale
    ) > 0.01:

        target_width = int(
            round(
                width * target_scale
            )
        )

        target_height = int(
            round(
                height * target_scale
            )
        )

        log(
            f"       Resizing output "
            f"to "
            f"{target_width}x"
            f"{target_height}..."
        )

        result = (
            Image
            .fromarray(output_np)
            .resize(
                (
                    target_width,
                    target_height,
                ),
                Image.LANCZOS,
            )
        )

    else:

        result = Image.fromarray(
            output_np
        )

    os.makedirs(
        os.path.dirname(
            os.path.abspath(
                output_path
            )
        ) or ".",
        exist_ok=True,
    )

    log(
        "       Saving output..."
    )

    result.save(
        output_path,
        "PNG",
    )

    elapsed = (
        time.time() - start
    )

    size_mb = (
        os.path.getsize(
            output_path
        ) / 1e6
    )

    return (
        result.size,
        elapsed,
        size_mb,
    )


# ============================================================
# BATCH
# ============================================================

def collect_images(root):

    tasks = []

    for directory, _, files in (
        os.walk(root)
    ):

        for filename in files:

            if (
                Path(filename)
                .suffix
                .lower()
                not in IMG_EXTS
            ):
                continue

            source = os.path.join(
                directory,
                filename,
            )

            relative = os.path.relpath(
                source,
                root,
            )

            tasks.append(
                (
                    source,
                    relative,
                )
            )

    return sorted(
        tasks,
        key=lambda x:
        x[1].lower(),
    )


def process_batch(
    tasks,
    input_root,
    output_root,
    model,
    model_scale,
    target_scale,
    device,
    tile_size,
    tile_pad,
):

    total = len(tasks)

    if total == 0:

        log(
            "[Real-ESRGAN] "
            "No images found."
        )

        return

    start = time.time()
    times = []

    for index, (
        source,
        relative,
    ) in enumerate(
        tasks,
        1,
    ):

        destination = os.path.join(
            output_root,
            os.path.splitext(
                relative
            )[0]
            + "_upscaled.png",
        )

        os.makedirs(
            os.path.dirname(
                destination
            ),
            exist_ok=True,
        )

        log("")
        log(
            f"[{index}/{total}] "
            f"STARTING"
        )

        (
            output_size,
            image_time,
            output_mb,
        ) = process_single(
            source,
            destination,
            model,
            model_scale,
            target_scale,
            device,
            tile_size,
            tile_pad,
        )

        times.append(
            image_time
        )

        completed = index
        remaining = (
            total - completed
        )

        average = (
            sum(times) / len(times)
        )

        eta = (
            average * remaining
        )

        log("")
        log(
            f"[{index}/{total}] "
            f"✓ DONE"
        )

        log(
            f"       Output size   : "
            f"{output_size[0]}x"
            f"{output_size[1]}"
        )

        log(
            f"       Output file   : "
            f"{output_mb:.2f} MB"
        )

        log(
            f"       Progress      : "
            f"{completed}/{total} "
            f"({completed / total * 100:.1f}%)"
        )

        log(
            f"       ETA           : "
            f"{format_seconds(eta)}"
        )

        log(
            f"       Elapsed       : "
            f"{format_seconds(time.time() - start)}"
        )

        print_gpu_memory()


# ============================================================
# ZIP
# ============================================================

def package_zip(
    source_dir,
    output_path,
):

    with zipfile.ZipFile(
        output_path,
        "w",
        zipfile.ZIP_DEFLATED,
    ) as archive:

        files = []

        for directory, _, names in (
            os.walk(source_dir)
        ):

            for name in names:

                path = os.path.join(
                    directory,
                    name,
                )

                relative = os.path.relpath(
                    path,
                    source_dir,
                )

                files.append(
                    (
                        path,
                        relative,
                    )
                )

        for index, (
            path,
            relative,
        ) in enumerate(
            files,
            1,
        ):

            archive.write(
                path,
                relative,
            )

            log(
                f"[ZIP] Packaged "
                f"{index}/{len(files)} "
                f"({index / len(files) * 100:.1f}%) "
                f": {relative}"
            )


def process_zip(
    input_path,
    output_path,
    model,
    model_scale,
    target_scale,
    device,
    tile_size,
    tile_pad,
):

    temp_input = tempfile.mkdtemp(
        prefix="realesrgan_in_"
    )

    temp_output = tempfile.mkdtemp(
        prefix="realesrgan_out_"
    )

    try:

        log("")
        log("=" * 70)
        log(
            "[Real-ESRGAN] "
            "ZIP PROCESSING"
        )
        log("=" * 70)

        with zipfile.ZipFile(
            input_path,
            "r",
        ) as archive:

            members = (
                archive.infolist()
            )

            for index, member in (
                enumerate(
                    members,
                    1,
                )
            ):

                if member.is_dir():
                    continue

                archive.extract(
                    member,
                    temp_input,
                )

                log(
                    f"[ZIP] Extracted "
                    f"{index}/{len(members)} "
                    f"({index / len(members) * 100:.1f}%)"
                )

        tasks = collect_images(
            temp_input
        )

        log(
            f"[ZIP] Found "
            f"{len(tasks)} image(s)"
        )

        process_batch(
            tasks,
            temp_input,
            temp_output,
            model,
            model_scale,
            target_scale,
            device,
            tile_size,
            tile_pad,
        )

        package_zip(
            temp_output,
            output_path,
        )

    finally:

        shutil.rmtree(
            temp_input,
            ignore_errors=True,
        )

        shutil.rmtree(
            temp_output,
            ignore_errors=True,
        )


# ============================================================
# MAIN
# ============================================================

def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input",
        required=True,
    )

    parser.add_argument(
        "--output",
        required=True,
    )

    parser.add_argument(
        "--scale",
        type=float,
        default=4.0,
    )

    parser.add_argument(
        "--device",
        default="cuda",
    )

    parser.add_argument(
        "--model-dir",
        required=True,
    )

    parser.add_argument(
        "--tile",
        type=int,
        default=512,
    )

    parser.add_argument(
        "--tile-pad",
        type=int,
        default=32,
    )

    args = parser.parse_args()

    if args.scale not in (
        2.0,
        3.5,
        4.0,
    ):
        raise SystemExit(
            "Scale must be "
            "2, 3.5 or 4."
        )

    if not os.path.exists(
        args.input
    ):
        raise SystemExit(
            f"Input not found: "
            f"{args.input}"
        )

    device = (
        "cuda"
        if args.device == "cuda"
        else "cpu"
    )

    if (
        device == "cuda"
        and not torch.cuda.is_available()
    ):
        raise SystemExit(
            "CUDA is not available."
        )

    model_name = (
        "RealESRGAN_x2plus.pth"
        if args.scale == 2.0
        else "RealESRGAN_x4plus.pth"
    )

    model_path = os.path.join(
        args.model_dir,
        model_name,
    )

    if not os.path.exists(
        model_path
    ):
        raise SystemExit(
            f"Model not found: "
            f"{model_path}"
        )

    log("")
    log("=" * 70)
    log(
        "🚀 REAL-ESRGAN START"
    )
    log("=" * 70)

    if device == "cuda":

        log(
            f"GPU    : "
            f"{torch.cuda.get_device_name(0)}"
        )

    start = time.time()

    log(
        "[Real-ESRGAN] "
        "Loading model..."
    )

    model, model_scale = load_model(
        model_path,
        device,
    )

    log(
        f"[Real-ESRGAN] "
        f"✓ Model loaded "
        f"in {time.time() - start:.2f}s"
    )

    print_gpu_memory()

    try:

        if os.path.isdir(
            args.input
        ):

            output_dir = (
                tempfile.mkdtemp(
                    prefix=
                    "realesrgan_folder_"
                )
            )

            try:

                tasks = collect_images(
                    args.input
                )

                process_batch(
                    tasks,
                    args.input,
                    output_dir,
                    model,
                    model_scale,
                    args.scale,
                    device,
                    args.tile,
                    args.tile_pad,
                )

                if args.output.lower().endswith(
                    ".zip"
                ):

                    package_zip(
                        output_dir,
                        args.output,
                    )

                else:

                    os.makedirs(
                        args.output,
                        exist_ok=True,
                    )

                    for item in os.listdir(
                        output_dir
                    ):

                        src = os.path.join(
                            output_dir,
                            item,
                        )

                        dst = os.path.join(
                            args.output,
                            item,
                        )

                        if os.path.isdir(
                            src
                        ):

                            shutil.copytree(
                                src,
                                dst,
                                dirs_exist_ok=True,
                            )

                        else:

                            shutil.copy2(
                                src,
                                dst,
                            )

            finally:

                shutil.rmtree(
                    output_dir,
                    ignore_errors=True,
                )

        elif args.input.lower().endswith(
            ".zip"
        ):

            process_zip(
                args.input,
                args.output,
                model,
                model_scale,
                args.scale,
                device,
                args.tile,
                args.tile_pad,
            )

        else:

            process_single(
                args.input,
                args.output,
                model,
                model_scale,
                args.scale,
                device,
                args.tile,
                args.tile_pad,
            )

        log("")
        log(
            "✅ REAL-ESRGAN COMPLETE"
        )

    finally:

        del model

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

            try:
                torch.cuda.synchronize()
            except Exception:
                pass

        log(
            "🧹 Model released and "
            "GPU memory cleaned."
        )


if __name__ == "__main__":
    main()
'''

with open(
    RUNNER_PATH,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        RUNNER_CODE
    )

print("\n✅ Runner created:")
print(RUNNER_PATH)

print("\n" + "=" * 70)
print("✅ REAL-ESRGAN WARM-UP COMPLETE")
print("=" * 70)
print(
    "Model weights are cached."
)
print(
    "Model is NOT loaded into GPU."
)
print(
    "GPU remains clean."
)
print("=" * 70)

# ============================================================
# 3/3 — CHATTERBOX TURBO
# ============================================================

print("\n" + "=" * 78)
print("🔊 3/3 — CHATTERBOX TURBO WARM-UP")
print("=" * 78)

# @title
# ============================================================
# CHATTERBOX TURBO WARM-UP — LIGHTWEIGHT
# Kaggle + Colab
#
# IMPORTANT:
# - Reuses the notebook's existing PyTorch/CUDA
# - Does NOT reinstall Torch
# - Does NOT install Gradio or unnecessary packages
# - Downloads Chatterbox Turbo model
# - Creates isolated runner
# ============================================================

import os
import sys
import subprocess

IS_KAGGLE = os.path.exists("/kaggle/working")

BASE_DIR = (
    "/kaggle/working"
    if IS_KAGGLE
    else "/content"
)

ENV_DIR = os.path.join(
    BASE_DIR,
    "chatterbox_env"
)

os.environ["PYTHONPATH"] = (
    ENV_DIR
    + os.pathsep
    + os.environ.get(
        "PYTHONPATH",
        ""
    )
)

if ENV_DIR not in sys.path:
    sys.path.insert(
        0,
        ENV_DIR
    )

MODEL_DIR = os.path.join(
    ENV_DIR,
    "models",
    "chatterbox-turbo"
)

RUNNER_PATH = os.path.join(
    BASE_DIR,
    "run_tts.py"
)

os.makedirs(
    ENV_DIR,
    exist_ok=True
)

print("=" * 70)
print("🚀 CHATTERBOX TURBO WARM-UP")
print("=" * 70)

print(
    f"Platform : "
    f"{'Kaggle' if IS_KAGGLE else 'Colab'}"
)

print(
    f"Python   : "
    f"{sys.version.split()[0]}"
)

print(
    f"Base     : {BASE_DIR}"
)

# ============================================================
# 1. USE EXISTING TORCH
# ============================================================

import torch

print(
    f"\n✅ PyTorch : {torch.__version__}"
)

print(
    f"✅ CUDA    : "
    f"{torch.cuda.is_available()}"
)

if torch.cuda.is_available():

    print(
        f"✅ GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )

# ============================================================
# 2. INSTALL ONLY TTS RUNTIME DEPENDENCIES
#
# IMPORTANT:
# --no-deps prevents pip from trying to build/resolve a
# massive dependency tree.
# ============================================================

print(
    "\n📦 Installing Chatterbox runtime packages..."
)

PACKAGES = [
    "numpy==2.2.6",
    "numba==0.67.0",
    "llvmlite==0.49.0",
    "librosa==0.11.0",
    "s3tokenizer",
    "transformers==5.2.0",
    "diffusers==0.29.0",
    "safetensors==0.5.3",
    "conformer==0.3.2",
    "pyloudnorm",
    "omegaconf",
    "onnx",
    "onnxruntime",
    "einops",
    "tokenizers==0.22.1",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--target",
        ENV_DIR,
        "--no-deps",
        "--no-cache-dir",
        "-q",
        *PACKAGES,
    ],
    check=True,
)

print(
    "✅ Runtime packages installed."
)

# ============================================================
# 3. RESEMBLE PERTH
# ============================================================

print(
    "\n📦 Installing resemble-perth..."
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--target",
        ENV_DIR,
        "--no-deps",
        "--no-cache-dir",
        "-q",
        "git+https://github.com/resemble-ai/Perth.git@master",
    ],
    check=True,
)

print(
    "✅ Perth ready."
)

# ============================================================
# 4. CHATTERBOX SOURCE
# ============================================================

print(
    "\n📦 Installing Chatterbox source..."
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--target",
        ENV_DIR,
        "--no-deps",
        "--no-cache-dir",
        "-q",
        "git+https://github.com/resemble-ai/chatterbox.git",
    ],
    check=True,
)

print(
    "✅ Chatterbox source ready."
)

# ============================================================
# 5. DOWNLOAD MODEL
# ============================================================

print(
    "\n⬇️ Checking Chatterbox Turbo model..."
)

sys.path.insert(
    0,
    ENV_DIR
)

os.environ["HF_HOME"] = os.path.join(
    ENV_DIR,
    "hf_cache"
)

from huggingface_hub import snapshot_download

if (
    not os.path.isdir(MODEL_DIR)
    or not os.listdir(MODEL_DIR)
):

    print(
        "⬇️ Downloading model..."
    )

    snapshot_download(
        "ResembleAI/chatterbox-turbo",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
        token=False
    )

    print(
        "✅ Model downloaded."
    )

else:

    print(
        "✅ Model already cached."
    )

# ============================================================
# 6. NUMPY COMPATIBILITY PATCHES
# ============================================================

print(
    "\n🔧 Applying compatibility patches..."
)

# ------------------------------------------------------------
# s3tokenizer
# ------------------------------------------------------------

s3_path = os.path.join(
    ENV_DIR,
    "chatterbox",
    "models",
    "s3tokenizer",
    "s3tokenizer.py"
)

if os.path.exists(s3_path):

    with open(
        s3_path,
        "r",
        encoding="utf-8"
    ) as f:

        content = f.read()

    old = (
        "mel_spec = self._mel_filters.to(self.device) "
        "@ magnitudes"
    )

    new = (
        "mel_spec = self._mel_filters.to(self.device).float() "
        "@ magnitudes.float()"
    )

    if old in content:

        content = content.replace(
            old,
            new
        )

        with open(
            s3_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(content)

        print(
            "✅ s3tokenizer patched."
        )

    else:

        print(
            "✅ s3tokenizer already patched."
        )

# ------------------------------------------------------------
# voice encoder
# ------------------------------------------------------------

voice_path = os.path.join(
    ENV_DIR,
    "chatterbox",
    "models",
    "voice_encoder",
    "voice_encoder.py"
)

if os.path.exists(voice_path):

    with open(
        voice_path,
        "r",
        encoding="utf-8"
    ) as f:

        content = f.read()

    content = content.replace(
        "_, (hidden, _) = self.lstm(mels)",
        "_, (hidden, _) = self.lstm(mels.float())"
    )

    content = content.replace(
        "utt_embeds = self.inference(mels.to(self.device), mel_lens",
        "utt_embeds = self.inference(mels.to(self.device).float(), mel_lens"
    )

    with open(
        voice_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(content)

    print(
        "✅ voice_encoder patched."
    )

# ============================================================
# 7. CREATE RUNNER
# ============================================================

RUNNER_CODE = r'''
#!/usr/bin/env python3

import os
import sys
import re
import argparse
import traceback

BASE_DIR = os.path.dirname(
    os.path.abspath(__file__)
)

ENV_DIR = os.path.join(
    BASE_DIR,
    "chatterbox_env"
)

MODEL_DIR = os.path.join(
    ENV_DIR,
    "models",
    "chatterbox-turbo"
)

# ------------------------------------------------------------
# Put our environment first, but KEEP SYSTEM TORCH available.
# ------------------------------------------------------------

if ENV_DIR not in sys.path:
    sys.path.insert(0, ENV_DIR)

import numpy as np
import soundfile as sf
import librosa
import torch


# ============================================================
# REFERENCE AUDIO
# ============================================================

def normalize_ref(
    input_path,
    output_path,
    sr
):

    data, _ = librosa.load(
        input_path,
        sr=sr
    )

    data, _ = librosa.effects.trim(
        data,
        top_db=20,
        frame_length=512,
        hop_length=128
    )

    peak = np.max(
        np.abs(data)
    )

    if peak > 0:
        data *= 0.9 / peak

    max_samples = 10 * sr

    if len(data) > max_samples:
        data = data[:max_samples]

    min_samples = int(
        5.1 * sr
    )

    if len(data) < min_samples:

        data = np.concatenate(
            [
                data,
                np.zeros(
                    min_samples - len(data),
                    dtype=np.float32
                )
            ]
        )

        print(
            f"⚠️ Reference padded to "
            f"{len(data) / sr:.1f}s",
            flush=True
        )

    sf.write(
        output_path,
        data,
        sr
    )


# ============================================================
# TEXT CHUNKING
# ============================================================

def smart_chunk(
    text,
    max_chars=200
):

    sentences = re.split(
        r'(?<=[.!?।\n])\s+',
        text.strip()
    )

    chunks = []
    current = ""

    for sentence in sentences:

        sentence = sentence.strip()

        if not sentence:
            continue

        if (
            len(current)
            + len(sentence)
            + 1
            <= max_chars
        ):

            current = (
                f"{current} {sentence}".strip()
                if current
                else sentence
            )

            continue

        if current:
            chunks.append(
                current
            )

        if len(sentence) <= max_chars:

            current = sentence

        else:

            current = ""

            for word in sentence.split():

                if (
                    len(current)
                    + len(word)
                    + 1
                    <= max_chars
                ):

                    current = (
                        f"{current} {word}".strip()
                        if current
                        else word
                    )

                else:

                    if current:
                        chunks.append(
                            current
                        )

                    current = word

    if current:
        chunks.append(
            current
        )

    return chunks


# ============================================================
# AUDIO CLEANUP
# ============================================================

def process_audio_chunk(
    wav,
    sr
):

    if isinstance(
        wav,
        torch.Tensor
    ):

        wav = (
            wav.detach()
            .cpu()
            .numpy()
        )

    if wav.ndim == 2:

        if wav.shape[0] == 1:
            wav = wav.squeeze(0)

        elif wav.shape[1] == 1:
            wav = wav.squeeze(1)

        else:
            wav = wav.mean(
                axis=0
            )

    wav = wav.astype(
        np.float32
    )

    fade = int(
        sr * 0.02
    )

    if len(wav) > fade * 2:

        wav[:fade] *= np.linspace(
            0,
            1,
            fade
        )

        wav[-fade:] *= np.linspace(
            1,
            0,
            fade
        )

    return wav


# ============================================================
# MODEL
# ============================================================

def load_model():

    from chatterbox.tts_turbo import (
        ChatterboxTurboTTS
    )

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"🚀 Loading ChatterboxTurboTTS "
        f"on {device}...",
        flush=True
    )

    model = (
        ChatterboxTurboTTS.from_local(
            MODEL_DIR,
            device=device
        )
    )

    print(
        f"✅ Model loaded | "
        f"SR: {model.sr} Hz",
        flush=True
    )

    return model


# ============================================================
# SINGLE
# ============================================================

def run_single(args):

    model = load_model()
    sr = model.sr

    ref_path = "/tmp/chatterbox_ref.wav"

    normalize_ref(
        args.ref_audio,
        ref_path,
        sr
    )

    with open(
        args.script,
        "r",
        encoding="utf-8"
    ) as f:

        text = f.read()

    chunks = smart_chunk(
        text,
        args.max_chars
    )

    print(
        f"🔹 Script split into "
        f"{len(chunks)} chunk(s)",
        flush=True
    )

    audio = []

    for index, chunk in enumerate(
        chunks,
        1
    ):

        print(
            f"   |█| Chunk "
            f"{index}/{len(chunks)}: "
            f"{chunk[:60]}",
            flush=True
        )

        try:

            wav = model.generate(
                chunk,
                audio_prompt_path=ref_path,
                temperature=args.temperature
            )

            wav = process_audio_chunk(
                wav,
                sr
            )

            if len(wav) < sr * 0.5:
                continue

            audio.append(
                wav
            )

            if index < len(chunks):

                audio.append(
                    np.zeros(
                        int(
                            sr * args.pause
                        ),
                        dtype=np.float32
                    )
                )

            print(
                f"   ✅ Chunk "
                f"{index} done.",
                flush=True
            )

        except Exception as e:

            print(
                f"   ❌ Chunk "
                f"{index} failed: {e}",
                flush=True
            )

            traceback.print_exc()

    if not audio:
        raise RuntimeError(
            "Generation failed."
        )

    sf.write(
        args.output,
        np.concatenate(audio),
        sr
    )

    print(
        f"✅ Saved: {args.output}",
        flush=True
    )


# ============================================================
# MULTI
# ============================================================

def run_multi(args):

    model = load_model()
    sr = model.sr

    with open(
        args.script,
        "r",
        encoding="utf-8"
    ) as f:

        script = f.read()

    segments = []

    for line in script.splitlines():

        line = line.strip()

        if (
            not line
            or ":" not in line
        ):
            continue

        tag, text = line.split(
            ":",
            1
        )

        tag = tag.strip()
        text = text.strip()

        if (
            tag.startswith("[")
            and tag.endswith("]")
        ):

            match = re.match(
                r"\[speaker_(\d+)\]",
                tag
            )

            speaker = (
                int(match.group(1))
                if match
                else 1
            )

        elif (
            len(tag) == 1
            and tag.isalpha()
        ):

            speaker = (
                ord(tag.upper())
                - ord("A")
                + 1
            )

        else:
            continue

        if text:

            segments.append(
                (
                    speaker,
                    text
                )
            )

    print(
        f"📝 Parsed "
        f"{len(segments)} segments",
        flush=True
    )

    speakers = {}

    for speaker_id in sorted(
        set(
            sid
            for sid, _ in segments
        )
    ):

        ref = os.path.join(
            args.refs_dir,
            f"speaker_{speaker_id}.wav"
        )

        if not os.path.exists(ref):

            alt = os.path.join(
                args.refs_dir,
                f"spk{speaker_id}.wav"
            )

            if not os.path.exists(alt):

                raise FileNotFoundError(
                    f"Missing reference "
                    f"for speaker "
                    f"{speaker_id}"
                )

            ref = alt

        normalized = os.path.join(
            "/tmp",
            f"spk{speaker_id}_norm.wav"
        )

        normalize_ref(
            ref,
            normalized,
            sr
        )

        speakers[
            speaker_id
        ] = normalized

        print(
            f"✅ Speaker "
            f"{speaker_id} ready",
            flush=True
        )

    audio = []

    for index, (
        speaker_id,
        text
    ) in enumerate(
        segments,
        1
    ):

        print(
            f"\n|█| Line "
            f"{index}/{len(segments)} "
            f"| Speaker "
            f"{speaker_id}: "
            f"{text[:60]}",
            flush=True
        )

        line_audio = []

        for chunk in smart_chunk(
            text,
            args.max_chars
        ):

            try:

                wav = model.generate(
                    chunk,
                    audio_prompt_path=
                    speakers[speaker_id],
                    temperature=args.temperature
                )

                wav = process_audio_chunk(
                    wav,
                    sr
                )

                if len(wav) >= sr * 0.5:

                    line_audio.append(
                        wav
                    )

            except Exception as e:

                print(
                    f"❌ Generation failed: "
                    f"{e}",
                    flush=True
                )

        if line_audio:

            audio.append(
                np.concatenate(
                    line_audio
                )
            )

            if index < len(segments):

                audio.append(
                    np.zeros(
                        int(
                            sr * args.pause
                        ),
                        dtype=np.float32
                    )
                )

            print(
                f"✅ Line "
                f"{index} synthesized.",
                flush=True
            )

    if not audio:

        raise RuntimeError(
            "Dialogue generation failed."
        )

    sf.write(
        args.output,
        np.concatenate(audio),
        sr
    )

    print(
        f"✅ Saved: {args.output}",
        flush=True
    )


# ============================================================
# MAIN
# ============================================================

def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--mode",
        choices=[
            "single",
            "multi"
        ],
        required=True
    )

    parser.add_argument(
        "--script",
        required=True
    )

    parser.add_argument(
        "--output",
        required=True
    )

    parser.add_argument(
        "--temperature",
        type=float,
        default=1.0
    )

    parser.add_argument(
        "--pause",
        type=float,
        default=0.5
    )

    parser.add_argument(
        "--max-chars",
        type=int,
        default=200
    )

    parser.add_argument(
        "--ref-audio"
    )

    parser.add_argument(
        "--refs-dir"
    )

    args = parser.parse_args()

    if (
        args.mode == "single"
        and not args.ref_audio
    ):

        parser.error(
            "--ref-audio required"
        )

    if (
        args.mode == "multi"
        and not args.refs_dir
    ):

        parser.error(
            "--refs-dir required"
        )

    if args.mode == "single":

        run_single(args)

    else:

        run_multi(args)


if __name__ == "__main__":

    try:

        main()

    finally:

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

            try:
                torch.cuda.synchronize()
            except Exception:
                pass

        print(
            "🧹 Chatterbox subprocess "
            "finished; GPU cache released.",
            flush=True
        )
'''

with open(
    RUNNER_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        RUNNER_CODE
    )

# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)
print("✅ CHATTERBOX WARM-UP COMPLETE")
print("=" * 70)

print(
    f"Environment : {ENV_DIR}"
)

print(
    f"Model       : {MODEL_DIR}"
)

print(
    f"Runner      : {RUNNER_PATH}"
)

print(
    "GPU         : Model NOT loaded"
)

print("=" * 70)

# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 78)
print("🚀 ALL 3 TOOLS — WARM-UP COMPLETE")
print("=" * 78)
print("✅ Whisper        : cached and GPU memory released")
print("✅ Real-ESRGAN    : models cached + runner created")
print("✅ Chatterbox     : model cached + runner created")
print("🧹 GPU            : warm-up models released")
print("=" * 78)



🔥 ALL 3 TOOLS — WARM-UP START

🎙️ 1/3 — WHISPER WARM-UP
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 46.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
🖥️ Device: CUDA
⬇️ Caching Whisper large-v3-turbo...


100%|██████████████████████████████████████| 1.51G/1.51G [00:08<00:00, 195MiB/s]


✅ Whisper warm-up complete!
✅ Model: large-v3-turbo
🧹 GPU memory released.

🖼️ 2/3 — REAL-ESRGAN WARM-UP
🚀 REAL-ESRGAN WARM-UP
Platform : Colab
Python   : 3.13.15
Base dir : /content

🔍 Checking existing dependencies...
✅ PyTorch : 2.11.0+cu128
✅ NumPy   : 2.1.3
✅ Pillow  : ready
✅ CUDA    : True

⬇️ Checking Real-ESRGAN models...
⬇️ Downloading RealESRGAN_x4plus.pth...
✅ Downloaded RealESRGAN_x4plus.pth (63.9 MB)
⬇️ Downloading RealESRGAN_x2plus.pth...
✅ Downloaded RealESRGAN_x2plus.pth (64.0 MB)

✅ Runner created:
/content/run_realesrgan.py

✅ REAL-ESRGAN WARM-UP COMPLETE
Model weights are cached.
Model is NOT loaded into GPU.
GPU remains clean.

🔊 3/3 — CHATTERBOX TURBO WARM-UP
🚀 CHATTERBOX TURBO WARM-UP
Platform : Colab
Python   : 3.13.15
Base     : /content

✅ PyTorch : 2.11.0+cu128
✅ CUDA    : True
✅ GPU     : Tesla T4

📦 Installing Chatterbox runtime packages...
✅ Runtime packages installed.

📦 Installing resemble-perth...
✅ Perth ready.

📦 Installing Chatterbox source...
✅ Chat

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Model downloaded.

🔧 Applying compatibility patches...
✅ s3tokenizer patched.
✅ voice_encoder patched.

✅ CHATTERBOX WARM-UP COMPLETE
Environment : /content/chatterbox_env
Model       : /content/chatterbox_env/models/chatterbox-turbo
Runner      : /content/run_tts.py
GPU         : Model NOT loaded

🚀 ALL 3 TOOLS — WARM-UP COMPLETE
✅ Whisper        : cached and GPU memory released
✅ Real-ESRGAN    : models cached + runner created
✅ Chatterbox     : model cached + runner created
🧹 GPU            : warm-up models released


In [ ]:
# @title AI Flows HUB (WEB UI)
# =============================================================================
# AI FLOWS HUB — ONE-CELL WEB UI
# Kaggle / Colab
#
# ONLY:
#   1. Chatterbox Turbo
#   2. Whisper
#   3. Real-ESRGAN
#
# Architecture:
# Browser -> Cloudflare HTTPS -> FastAPI -> child subprocess -> Google Drive
# Download: Browser -> Google Drive (direct)
# =============================================================================

import os
import re
import sys
import time
import uuid
import socket
import shutil
import zipfile
import asyncio
import threading
import subprocess
import json
import importlib.util
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path

# =============================================================================
# 1. WEB DEPENDENCIES
# =============================================================================

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart", "google-api-python-client", "google-auth-httplib2", "google-auth-oauthlib"],
    check=True,
)

from fastapi import FastAPI, File, Form, UploadFile, WebSocket
from fastapi.responses import HTMLResponse, JSONResponse
import uvicorn

# =============================================================================
# 2. RUNTIME PATHS
# =============================================================================

BASE_DIR = Path("/kaggle/working" if os.path.exists("/kaggle/working") else "/content")

CHATTERBOX_RUNNER = BASE_DIR / "run_tts.py"
WHISPER_RUNNER = BASE_DIR / "run_whisper.py"
UPSCALER_RUNNER = BASE_DIR / "run_realesrgan.py"

CHATTERBOX_ENV = BASE_DIR / "chatterbox_env"
WHISPER_ENV = BASE_DIR / "whisper_env"
UPSCALER_ENV = BASE_DIR / "realesrgan_env"
UPSCALER_MODELS = BASE_DIR / "realesrgan_models"

UPLOAD_DIR = BASE_DIR / "three_tools_uploads"
JOB_DIR = BASE_DIR / "three_tools_jobs"
CLOUDFLARED = BASE_DIR / "cloudflared"

# Google Drive (the notebook should already have: drive.mount('/content/drive'))
DRIVE_MOUNT = BASE_DIR / "drive" / "MyDrive"
DRIVE_FOLDER_NAME = "AI Flows Hub Downloads"
DRIVE_MIME = "application/vnd.google-apps.folder"
DRIVE_PERM = {"type": "anyone", "role": "reader"}

if not DRIVE_MOUNT.exists():
    raise RuntimeError(
        f"Google Drive is not mounted at {DRIVE_MOUNT}. Run drive.mount('/content/drive') first, then rerun this cell."
    )

try:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()
except Exception as exc:
    raise RuntimeError(f"Google account authentication failed: {exc}") from exc

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.auth import default as google_auth_default

GOOGLE_CREDS, _ = google_auth_default()


def drive_service():
    return build("drive", "v3", credentials=GOOGLE_CREDS, cache_discovery=False)


def ensure_drive_folder():
    service = drive_service()
    q = (
        f"name = '{DRIVE_FOLDER_NAME.replace(chr(39), chr(92) + chr(39))}' "
        f"and mimeType = '{DRIVE_MIME}' and trashed = false"
    )
    res = service.files().list(q=q, spaces="drive", fields="files(id,name)", pageSize=10).execute()
    files = res.get("files", [])
    if files:
        return files[0]["id"]
    meta = {"name": DRIVE_FOLDER_NAME, "mimeType": DRIVE_MIME}
    return service.files().create(body=meta, fields="id").execute()["id"]


DRIVE_FOLDER_ID = ensure_drive_folder()
print(f"✅ Google Drive ready: MyDrive/{DRIVE_FOLDER_NAME}", flush=True)

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
JOB_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 3. CONSTANTS / VALIDATION
# =============================================================================

CHATTERBOX_AUDIO = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
WHISPER_AUDIO = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".wma"}
IMAGES = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
SCALES = {2.0, 3.5, 4.0}

TOOL_META = {
    "chatterbox": {
        "name": "Chatterbox Turbo",
        "short": "Text to Speech",
        "runner": CHATTERBOX_RUNNER,
        "env": CHATTERBOX_ENV,
    },
    "whisper": {
        "name": "Whisper",
        "short": "Speech to Text",
        "runner": WHISPER_RUNNER,
        "env": WHISPER_ENV,
    },
    "upscaler": {
        "name": "Real-ESRGAN",
        "short": "Image Upscaling",
        "runner": UPSCALER_RUNNER,
        "env": UPSCALER_ENV,
    },
}


def safe_name(name: str) -> str:
    name = os.path.basename(name or "upload")
    name = re.sub(r"[^A-Za-z0-9._()\- ]+", "_", name)
    return name or "upload"


def runner_ready(path: Path) -> bool:
    return path.is_file()


WHISPER_RUNNER_VERSION = "2.0.0"

WHISPER_RUNNER_CODE = r'''#!/usr/bin/env python3
# RUNNER_VERSION = "2.0.0"
import argparse
import json
import sys
from pathlib import Path
import torch
import whisper

def srt_time(seconds):
    total_ms = round(float(seconds) * 1000)
    ms = total_ms % 1000
    total_seconds = total_ms // 1000
    hours = total_seconds // 3600
    minutes = (total_seconds // 60) % 60
    secs = total_seconds % 60
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"

def main():
    model = None
    try:
        p = argparse.ArgumentParser()
        p.add_argument("--audio", required=True)
        p.add_argument("--output-words", required=True)
        p.add_argument("--output-segments", required=True)
        p.add_argument("--output-segments-txt", required=True)
        p.add_argument("--output-words-txt", required=True)
        p.add_argument("--output-segment-srt", required=True)
        p.add_argument("--output-word-srt", required=True)
        p.add_argument("--language", default="auto")
        p.add_argument("--device", default="cuda")
        args = p.parse_args()

        device = "cuda" if args.device == "cuda" and torch.cuda.is_available() else "cpu"
        language = None if args.language in ("auto", "", "none", "null") else args.language

        print(f"[Whisper] Device: {device.upper()}", flush=True)
        print("[Whisper] Loading model: large-v3-turbo", flush=True)
        model = whisper.load_model("large-v3-turbo", device=device)
        print("[Whisper] Model loaded", flush=True)

        result = model.transcribe(
            args.audio,
            language=language,
            task="transcribe",
            verbose=True,
            condition_on_previous_text=True,
            temperature=0.0,
            best_of=5,
            beam_size=5,
            word_timestamps=True,
        )

        segments = result.get("segments", [])
        words = []

        for seg in segments:
            for w in seg.get("words", []) or []:
                text = str(w.get("word", "")).strip()
                if text:
                    words.append({
                        "word": text,
                        "start": float(w.get("start", 0.0)),
                        "end": float(w.get("end", 0.0)),
                    })

        seg_out = []
        for seg in segments:
            text = str(seg.get("text", "")).strip()
            if text:
                seg_out.append({
                    "id": seg.get("id", len(seg_out)),
                    "start": float(seg.get("start", 0.0)),
                    "end": float(seg.get("end", 0.0)),
                    "text": text,
                })

        with open(args.output_words, "w", encoding="utf-8") as f:
            json.dump(words, f, indent=2, ensure_ascii=False)

        with open(args.output_segments, "w", encoding="utf-8") as f:
            json.dump(seg_out, f, indent=2, ensure_ascii=False)

        with open(args.output_segments_txt, "w", encoding="utf-8") as f:
            for seg in seg_out:
                f.write(
                    f"[{float(seg['start']):.2f}s → {float(seg['end']):.2f}s] "
                    f"{seg['text']}\n"
                )

        with open(args.output_words_txt, "w", encoding="utf-8") as f:
            for word in words:
                f.write(
                    f"[{float(word['start']):.2f}s → {float(word['end']):.2f}s] "
                    f"{word['word']}\n"
                )

        with open(args.output_segment_srt, "w", encoding="utf-8") as f:
            n = 1
            for seg in seg_out:
                f.write(
                    f"{n}\n"
                    f"{srt_time(seg['start'])} --> {srt_time(seg['end'])}\n"
                    f"{seg['text']}\n\n"
                )
                n += 1

        with open(args.output_word_srt, "w", encoding="utf-8") as f:
            n = 1
            for word in words:
                f.write(
                    f"{n}\n"
                    f"{srt_time(word['start'])} --> {srt_time(word['end'])}\n"
                    f"{word['word']}\n\n"
                )
                n += 1

        detected = result.get("language", "unknown")
        print(f"Detected language: {detected} (prob=1.0)", flush=True)
        print(f"Segments: {len(seg_out)} | Words: {len(words)}", flush=True)
        print(f"Words JSON: {args.output_words}", flush=True)
        print(f"Segments JSON: {args.output_segments}", flush=True)
        print(f"Segments TXT: {args.output_segments_txt}", flush=True)
        print(f"Words TXT: {args.output_words_txt}", flush=True)
        print(f"Segment SRT: {args.output_segment_srt}", flush=True)
        print(f"Word SRT: {args.output_word_srt}", flush=True)
        print("Done", flush=True)

    finally:
        if model is not None:
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            try:
                torch.cuda.synchronize()
            except Exception:
                pass
        print("[Whisper] GPU memory released.", flush=True)

if __name__ == "__main__":
    main()
'''


def ensure_whisper_runner():
    needs_rewrite = True
    if WHISPER_RUNNER.is_file():
        try:
            content = WHISPER_RUNNER.read_text(encoding="utf-8")
            if f'RUNNER_VERSION = "{WHISPER_RUNNER_VERSION}"' in content:
                needs_rewrite = False
        except Exception:
            needs_rewrite = True

    if needs_rewrite:
        WHISPER_RUNNER.write_text(WHISPER_RUNNER_CODE, encoding="utf-8")
        os.chmod(WHISPER_RUNNER, 0o755)
        print(f"✅ Synchronized Whisper runner (v{WHISPER_RUNNER_VERSION}): {WHISPER_RUNNER}", flush=True)
    else:
        print(f"✅ Whisper runner is up-to-date (v{WHISPER_RUNNER_VERSION}): {WHISPER_RUNNER}", flush=True)


def verify_whisper_runner():
    ensure_whisper_runner()
    res = subprocess.run(
        [sys.executable, str(WHISPER_RUNNER), "--help"],
        check=True,
        capture_output=True,
        text=True,
    )
    help_text = res.stdout + res.stderr
    required_args = [
        "--output-segments-txt",
        "--output-words-txt",
        "--output-segment-srt",
        "--output-word-srt",
    ]
    missing = [arg for arg in required_args if arg not in help_text]
    if missing:
        raise RuntimeError(
            f"Whisper runner verification failed! Missing options: {missing}\nHelp text:\n{help_text}"
        )
    print("✅ Verified Whisper runner CLI arguments successfully.", flush=True)


def build_env(tool: str):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    if tool == "chatterbox":
        if CHATTERBOX_ENV.exists():
            env["PYTHONPATH"] = str(CHATTERBOX_ENV) + (os.pathsep + env.get("PYTHONPATH", "") if env.get("PYTHONPATH") else "")
        env["HF_HOME"] = str(CHATTERBOX_ENV / "hf_cache")
    elif tool == "upscaler":
        if UPSCALER_ENV.exists():
            env["PYTHONPATH"] = str(UPSCALER_ENV) + (os.pathsep + env.get("PYTHONPATH", "") if env.get("PYTHONPATH") else "")
    return env


verify_whisper_runner()

# =============================================================================
# 4. JOB ENGINE
# =============================================================================

WARM_WORKER_CODE = '#!/usr/bin/env python3\nimport os\nimport sys\nimport json\nimport traceback\nimport importlib.util\nfrom contextlib import redirect_stdout, redirect_stderr\n\nTOOL, RUNNER, BASE_DIR, CHATTERBOX_ENV, UPSCALER_ENV, UPSCALER_MODELS = sys.argv[1:7]\n\nclass PrefixWriter:\n    # stdout is redirected to this object during a job. Never call print()\n    # from write()/flush(); that would recursively call PrefixWriter.write().\n    def __init__(self, job_id):\n        self.job_id = job_id\n        self.buf = ""\n        self.stream = sys.__stdout__\n\n    def _emit(self, line):\n        self.stream.write(f"@@LOG|{self.job_id}|{line}\\n")\n        self.stream.flush()\n\n    def write(self, data):\n        if not data:\n            return 0\n        self.buf += str(data)\n        while "\\n" in self.buf:\n            line, self.buf = self.buf.split("\\n", 1)\n            line = line.rstrip("\\r")\n            if line:\n                self._emit(line)\n        return len(data)\n\n    def flush(self):\n        if self.buf:\n            self._emit(self.buf.rstrip("\\r"))\n            self.buf = ""\n\ndef load_module(path, name, env=None):\n    if env and os.path.isdir(env):\n        sys.path.insert(0, env)\n    spec = importlib.util.spec_from_file_location(name, path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError(f"Could not load runner: {path}")\n    mod = importlib.util.module_from_spec(spec)\n    sys.modules[name] = mod\n    spec.loader.exec_module(mod)\n    return mod\n\ndef boot(msg):\n    print(f"@@BOOT|{msg}", flush=True)\n\n\ndef chatterbox_generate_single(module, model, args):\n    """Use the already-loaded model directly. Never calls load_model()."""\n    import numpy as np\n    import soundfile as sf\n\n    sr = model.sr\n    ref_path = "/tmp/afh_chatterbox_ref.wav"\n    module.normalize_ref(args.ref_audio, ref_path, sr)\n    dur = len(sf.read(ref_path)[0]) / sr\n    print(f"✅ Reference ready ({dur:.1f}s)", flush=True)\n\n    with open(args.script, "r", encoding="utf-8") as f:\n        script_text = f.read()\n    chunks = module.smart_chunk(script_text, args.max_chars)\n    print(f"🔹 Script split into {len(chunks)} chunk(s)", flush=True)\n\n    audio_segs = []\n    for idx, chunk in enumerate(chunks):\n        print(f"   |█| Chunk {idx+1}/{len(chunks)}: {chunk[:60]}{\'...\' if len(chunk)>60 else \'\'}", flush=True)\n        try:\n            wav = model.generate(chunk, audio_prompt_path=ref_path, temperature=args.temperature)\n            wav = module.process_audio_chunk(wav, sr)\n            if len(wav) < sr * 0.5:\n                print(f"   ⚠️ Chunk {idx+1} produced empty audio!", flush=True)\n                continue\n            audio_segs.append(wav)\n            if idx < len(chunks) - 1:\n                audio_segs.append(np.zeros(int(sr * args.pause), dtype=np.float32))\n            print(f"   ✅ Chunk {idx+1} done.", flush=True)\n        except Exception as e:\n            print(f"   ❌ Chunk {idx+1} failed: {e}", flush=True)\n            traceback.print_exc()\n\n    if not audio_segs:\n        raise RuntimeError("Generation failed.")\n    sf.write(args.output, np.concatenate(audio_segs), sr)\n    print(f"✅ Saved: {args.output}", flush=True)\n\n\ndef chatterbox_generate_multi(module, model, args):\n    """Multi-speaker generation using the persistent model directly."""\n    import os\n    import re\n    import numpy as np\n    import soundfile as sf\n\n    sr = model.sr\n    with open(args.script, "r", encoding="utf-8") as f:\n        script_text = f.read()\n\n    segments = []\n    for line in script_text.strip().splitlines():\n        line = line.strip()\n        if not line or ":" not in line:\n            continue\n        tag, text = line.split(":", 1)\n        tag, text = tag.strip(), text.strip()\n        if tag.startswith("[") and tag.endswith("]"):\n            m = re.match(r"\\[speaker_(\\d+)\\]", tag, re.I)\n            spk_id = int(m.group(1)) if m else 1\n        elif len(tag) == 1 and tag.isalpha():\n            spk_id = ord(tag.upper()) - ord("A") + 1\n        else:\n            continue\n        if text:\n            segments.append((spk_id, text))\n\n    print(f"📝 Parsed {len(segments)} segments", flush=True)\n    counts = {}\n    for sid, _ in segments:\n        counts[sid] = counts.get(sid, 0) + 1\n    print(f"📊 Speaker distribution: {counts}", flush=True)\n\n    speaker_map = {}\n    for spk_id in sorted(set(s for s, _ in segments)):\n        ref_file = os.path.join(args.refs_dir, f"speaker_{spk_id}.wav")\n        if not os.path.exists(ref_file):\n            alt = os.path.join(args.refs_dir, f"spk{spk_id}.wav")\n            if os.path.exists(alt):\n                ref_file = alt\n            else:\n                raise FileNotFoundError(f"Missing reference for speaker {spk_id}")\n        out_ref = f"/tmp/afh_spk{spk_id}_norm.wav"\n        module.normalize_ref(ref_file, out_ref, sr)\n        dur = len(sf.read(out_ref)[0]) / sr\n        print(f"✅ Speaker {spk_id} ready ({dur:.1f}s)", flush=True)\n        speaker_map[spk_id] = out_ref\n\n    audio_segs = []\n    for idx, (spk_id, text) in enumerate(segments):\n        print(f"|█| Line {idx+1}/{len(segments)} | Speaker {spk_id}: {text[:60]}{\'...\' if len(text)>60 else \'\'}", flush=True)\n        ref_audio = speaker_map[spk_id]\n        line_audio = []\n        sub_chunks = module.smart_chunk(text, args.max_chars)\n        for cidx, chunk in enumerate(sub_chunks):\n            try:\n                wav = model.generate(chunk, audio_prompt_path=ref_audio, temperature=args.temperature)\n                wav = module.process_audio_chunk(wav, sr)\n                if len(wav) < sr * 0.5:\n                    print(f"   ⚠️ Sub-chunk {cidx+1} produced empty audio!", flush=True)\n                    continue\n                line_audio.append(wav)\n            except Exception as e:\n                print(f"   ❌ Sub-chunk {cidx+1} failed: {e}", flush=True)\n                traceback.print_exc()\n        if line_audio:\n            audio_segs.append(np.concatenate(line_audio))\n            if idx < len(segments) - 1:\n                audio_segs.append(np.zeros(int(sr * args.pause), dtype=np.float32))\n            print(f"✅ Line {idx+1} synthesized.", flush=True)\n        else:\n            raise RuntimeError(f"Line {idx+1} failed entirely.")\n\n    if not audio_segs:\n        raise RuntimeError("Dialogue generation failed.")\n    sf.write(args.output, np.concatenate(audio_segs), sr)\n    print(f"✅ Saved: {args.output}", flush=True)\n\n\ndef whisper_transcribe(module, model, args):\n    """Direct Whisper transcription; deliberately bypasses runner.main() so its\n    per-process cleanup cannot interfere with the persistent model."""\n    language = None if args.get("language") in (None, "", "auto", "none", "null") else args.get("language")\n    audio = args["audio"]\n    print(f"[Whisper] Device: {\'CUDA\' if module.torch.cuda.is_available() else \'CPU\'}", flush=True)\n    print("[Whisper] Using persistent large-v3-turbo model", flush=True)\n    result = model.transcribe(\n        audio,\n        language=language,\n        task="transcribe",\n        verbose=True,\n        condition_on_previous_text=True,\n        temperature=0.0,\n        best_of=5,\n        beam_size=5,\n        word_timestamps=True,\n    )\n    segments = result.get("segments", [])\n    words = []\n    for seg in segments:\n        for w in seg.get("words", []) or []:\n            text = str(w.get("word", "")).strip()\n            if text:\n                words.append({"word": text, "start": float(w.get("start", 0.0)), "end": float(w.get("end", 0.0))})\n    seg_out = []\n    for seg in segments:\n        text = str(seg.get("text", "")).strip()\n        if text:\n            seg_out.append({"id": seg.get("id", len(seg_out)), "start": float(seg.get("start", 0.0)), "end": float(seg.get("end", 0.0)), "text": text})\n\n    def srt_time(seconds):\n        total_ms = round(float(seconds) * 1000)\n        ms = total_ms % 1000\n        total_seconds = total_ms // 1000\n        return f"{total_seconds // 3600:02d}:{(total_seconds // 60) % 60:02d}:{total_seconds % 60:02d},{ms:03d}"\n\n    with open(args["output_words"], "w", encoding="utf-8") as f:\n        json.dump(words, f, indent=2, ensure_ascii=False)\n    with open(args["output_segments"], "w", encoding="utf-8") as f:\n        json.dump(seg_out, f, indent=2, ensure_ascii=False)\n    with open(args["output_segments_txt"], "w", encoding="utf-8") as f:\n        for seg in seg_out:\n            f.write(f"[{float(seg[\'start\']):.2f}s → {float(seg[\'end\']):.2f}s] {seg[\'text\']}\\n")\n    with open(args["output_words_txt"], "w", encoding="utf-8") as f:\n        for word in words:\n            f.write(f"[{float(word[\'start\']):.2f}s → {float(word[\'end\']):.2f}s] {word[\'word\']}\\n")\n    with open(args["output_segment_srt"], "w", encoding="utf-8") as f:\n        for n, seg in enumerate(seg_out, 1):\n            f.write(f"{n}\\n{srt_time(seg[\'start\'])} --> {srt_time(seg[\'end\'])}\\n{seg[\'text\']}\\n\\n")\n    with open(args["output_word_srt"], "w", encoding="utf-8") as f:\n        for n, word in enumerate(words, 1):\n            f.write(f"{n}\\n{srt_time(word[\'start\'])} --> {srt_time(word[\'end\'])}\\n{word[\'word\']}\\n\\n")\n\n    print(f"Detected language: {result.get(\'language\', \'unknown\')} (persistent)", flush=True)\n    print(f"Segments: {len(seg_out)} | Words: {len(words)}", flush=True)\n    print("✅ Whisper transcription complete", flush=True)\n\nmodule = None\nmodel = None\nmodel_native_scale = None\n\ntry:\n    if TOOL == "chatterbox":\n        os.environ["PYTHONPATH"] = CHATTERBOX_ENV + os.pathsep + os.environ.get("PYTHONPATH", "")\n        os.environ["HF_HOME"] = os.path.join(CHATTERBOX_ENV, "hf_cache")\n        module = load_module(RUNNER, "afh_cb_warm_fixed", CHATTERBOX_ENV)\n        boot("Loading Chatterbox Turbo into GPU...")\n        model = module.load_model()\n        boot("Chatterbox Turbo MODEL WARM & READY")\n\n    elif TOOL == "whisper":\n        module = load_module(RUNNER, "afh_whisper_warm_fixed")\n        device = "cuda" if module.torch.cuda.is_available() else "cpu"\n        boot("Loading Whisper large-v3-turbo into GPU...")\n        model = module.whisper.load_model("large-v3-turbo", device=device)\n        boot("Whisper large-v3-turbo MODEL WARM & READY")\n\n    elif TOOL == "upscaler":\n        if os.path.isdir(UPSCALER_ENV):\n            sys.path.insert(0, UPSCALER_ENV)\n        module = load_module(RUNNER, "afh_esrgan_warm_fixed", UPSCALER_ENV)\n        device = "cuda" if module.torch.cuda.is_available() else "cpu"\n        boot("Loading Real-ESRGAN 4x into GPU...")\n        path = os.path.join(UPSCALER_MODELS, "RealESRGAN_x4plus.pth")\n        if not os.path.isfile(path):\n            alt = os.path.join(UPSCALER_ENV, "models", "RealESRGAN_x4plus.pth")\n            if os.path.isfile(alt): path = alt\n        if not os.path.isfile(path): raise FileNotFoundError(f"Real-ESRGAN model not found: {path}")\n        model, model_native_scale = module.load_model(path, device)\n        boot("Real-ESRGAN 4x MODEL WARM & READY")\n    else:\n        raise RuntimeError(f"Unknown warm worker: {TOOL}")\n\n    print("@@READY", flush=True)\n\n    for raw in sys.stdin:\n        raw = raw.strip()\n        if not raw: continue\n        cmd = json.loads(raw)\n        if cmd.get("op") == "stop":\n            print("@@STOPPING", flush=True)\n            break\n        if cmd.get("op") != "job": continue\n\n        job_id = cmd["job_id"]\n        args = cmd.get("args", {})\n        print(f"@@JOB_START|{job_id}", flush=True)\n        writer = PrefixWriter(job_id)\n        try:\n            with redirect_stdout(writer), redirect_stderr(writer):\n                if TOOL == "chatterbox":\n                    ns = type("Args", (), {})()\n                    for k, v in args.items(): setattr(ns, k, v)\n                    if ns.mode == "single": chatterbox_generate_single(module, model, ns)\n                    else: chatterbox_generate_multi(module, model, ns)\n\n                elif TOOL == "whisper":\n                    whisper_transcribe(module, model, args)\n\n                else:\n                    scale = float(args["scale"])\n                    device = "cuda" if module.torch.cuda.is_available() else "cpu"\n                    requested_native = 2 if scale == 2.0 else 4\n                    if model is None or model_native_scale != requested_native:\n                        if model is not None:\n                            del model; model = None\n                            if module.torch.cuda.is_available():\n                                module.torch.cuda.empty_cache()\n                        name = "RealESRGAN_x2plus.pth" if requested_native == 2 else "RealESRGAN_x4plus.pth"\n                        path = os.path.join(UPSCALER_MODELS, name)\n                        if not os.path.isfile(path):\n                            alt = os.path.join(UPSCALER_ENV, "models", name)\n                            if os.path.isfile(alt): path = alt\n                        if not os.path.isfile(path): raise FileNotFoundError(path)\n                        print(f"Loading Real-ESRGAN {scale:g}x model into GPU...", flush=True)\n                        model, model_native_scale = module.load_model(path, device)\n                        print(f"Real-ESRGAN {scale:g}x model ready.", flush=True)\n\n                    module.log = lambda msg="": print(msg, flush=True)\n                    input_path, output_path = args["input"], args["output"]\n                    if os.path.isdir(input_path):\n                        module.process_folder(input_path, output_path, model, model_native_scale, scale, device, 512, 32)\n                    elif input_path.lower().endswith(".zip"):\n                        module.process_zip(input_path, output_path, model, model_native_scale, scale, device, 512, 32)\n                    else:\n                        module.process_single(input_path, output_path, model, model_native_scale, scale, device, 512, 32)\n\n            writer.flush()\n            print(f"@@JOB_DONE|{job_id}", flush=True)\n        except Exception as exc:\n            writer.flush()\n            print(f"@@JOB_ERROR|{job_id}|{type(exc).__name__}: {exc}", flush=True)\n            traceback.print_exc(file=sys.stdout)\nfinally:\n    try:\n        if model is not None:\n            del model\n        if module is not None and hasattr(module, "torch") and module.torch.cuda.is_available():\n            module.torch.cuda.empty_cache()\n    except Exception:\n        pass\n    print("@@EXITED", flush=True)\n'

class Job:
    def __init__(self, tool: str, filename: str):
        self.id = uuid.uuid4().hex[:12]
        self.tool = tool
        self.filename = filename
        self.created_at = time.time()
        self.started_at = None
        self.finished_at = None
        self.status = "queued"
        self.return_code = None
        self.process = None
        self.events = []
        self.output_path = None
        self.input_path = None
        self.words_path = None
        self.segments_path = None
        self.segments_txt_path = None
        self.words_txt_path = None
        self.segment_srt_path = None
        self.word_srt_path = None
        self.drive_downloads = {}
        self.lock = threading.Lock()
        self.warm = False

jobs = {}
jobs_lock = threading.Lock()
tool_locks = {k: threading.Lock() for k in TOOL_META}

def emit(job: Job, kind: str, message: str = "", **extra):
    event = {"type": kind, "message": message, "time": time.time(), **extra}
    with job.lock:
        job.events.append(event)

def mark_done(job: Job, status: str, message: str, code=None, **extra):
    job.status = status
    job.return_code = code
    job.finished_at = time.time()
    emit(job, "status", message, status=status, return_code=code, **extra)

def upload_file_to_drive(path: Path, filename: str | None = None) -> str:
    if not path or not path.is_file():
        raise FileNotFoundError(f"Output file not found: {path}")
    service = drive_service()
    name = filename or path.name
    created = service.files().create(
        body={"name": name, "parents": [DRIVE_FOLDER_ID]},
        media_body=MediaFileUpload(str(path), resumable=True),
        fields="id,name,size,webContentLink",
        supportsAllDrives=True,
    ).execute()
    file_id = created["id"]
    service.permissions().create(
        fileId=file_id, body=DRIVE_PERM, fields="id", supportsAllDrives=True
    ).execute()
    return f"https://drive.google.com/uc?export=download&id={file_id}"

def upload_job_outputs(job: Job):
    outputs = []
    if job.tool == "chatterbox" and job.output_path:
        outputs = [("audio", job.output_path)]
    elif job.tool == "upscaler" and job.output_path:
        outputs = [("output", job.output_path)]
    elif job.tool == "whisper":
        outputs = [("words", job.words_path), ("segments", job.segments_path), ("segments-txt", job.segments_txt_path), ("words-txt", job.words_txt_path), ("segment-srt", job.segment_srt_path), ("word-srt", job.word_srt_path)]
    links = {}
    for key, path in outputs:
        emit(job, "log", f"☁ Uploading to Google Drive: {path.name}")
        links[key] = upload_file_to_drive(path)
        emit(job, "log", f"☁ Google Drive ready: {path.name}")
    job.drive_downloads = links
    return links

WARM_WORKER_DIR = JOB_DIR / "warm_workers"
WARM_WORKER_DIR.mkdir(parents=True, exist_ok=True)
WARM_TEMPL = WARM_WORKER_DIR / "persistent_gpu_worker.py"
WARM_TEMPL.write_text(WARM_WORKER_CODE, encoding="utf-8")
os.chmod(WARM_TEMPL, 0o755)
WARM_STATE = {tool: {"enabled": False, "state": "off", "message": "Cold mode", "pid": None} for tool in TOOL_META}
WARM_PROCS = {}
WARM_CURRENT = {}
WARM_LOCKS = {tool: threading.RLock() for tool in TOOL_META}

def warm_state_snapshot():
    return {k: dict(v) for k, v in WARM_STATE.items()}

def set_warm_state(tool, state, message=None, enabled=None):
    with WARM_LOCKS[tool]:
        if enabled is not None:
            WARM_STATE[tool]["enabled"] = bool(enabled)
        WARM_STATE[tool]["state"] = state
        if message is not None:
            WARM_STATE[tool]["message"] = message
        p = WARM_PROCS.get(tool)
        WARM_STATE[tool]["pid"] = p.pid if p else None

def _warm_reader(tool, proc):
    try:
        for raw in proc.stdout or []:
            line = raw.rstrip("\r\n")
            if not line:
                continue
            if line == "@@READY":
                set_warm_state(tool, "ready", f"{TOOL_META[tool]['name']} model ready")
            elif line.startswith("@@BOOT|"):
                msg = line.split("|", 1)[1]
                set_warm_state(tool, "ready" if "READY" in msg.upper() else "loading", msg)
            elif line.startswith("@@JOB_START|"):
                continue
            elif line.startswith("@@LOG|"):
                _, job_id, msg = line.split("|", 2)
                with WARM_LOCKS[tool]:
                    cur = WARM_CURRENT.get(tool)
                    if cur and cur["job"].id == job_id:
                        emit(cur["job"], "log", msg)
            elif line.startswith("@@JOB_DONE|"):
                job_id = line.split("|", 1)[1]
                with WARM_LOCKS[tool]:
                    cur = WARM_CURRENT.get(tool)
                    if cur and cur["job"].id == job_id:
                        cur["ok"] = True
                        cur["event"].set()
            elif line.startswith("@@JOB_ERROR|"):
                _, job_id, msg = line.split("|", 2)
                with WARM_LOCKS[tool]:
                    cur = WARM_CURRENT.get(tool)
                    if cur and cur["job"].id == job_id:
                        cur["error"] = msg
                        cur["event"].set()
            elif line == "@@EXITED":
                if WARM_STATE[tool]["enabled"]:
                    set_warm_state(tool, "error", "Warm worker exited unexpectedly")
                else:
                    set_warm_state(tool, "off", "Cold mode")
    except Exception as exc:
        set_warm_state(tool, "error", f"Warm worker reader error: {exc}")
    finally:
        with WARM_LOCKS[tool]:
            WARM_PROCS.pop(tool, None)
            cur = WARM_CURRENT.get(tool)
            if cur and not cur["event"].is_set():
                cur["error"] = "Warm worker exited unexpectedly."
                cur["event"].set()

def start_warm_worker(tool: str):
    with WARM_LOCKS[tool]:
        p = WARM_PROCS.get(tool)
        if p and p.poll() is None:
            return
        set_warm_state(tool, "loading", f"Loading {TOOL_META[tool]['name']} into GPU...", enabled=True)
        p = subprocess.Popen(
            [sys.executable, "-u", str(WARM_TEMPL), tool, str(TOOL_META[tool]["runner"]), str(BASE_DIR), str(CHATTERBOX_ENV), str(UPSCALER_ENV), str(UPSCALER_MODELS)],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True, env=build_env(tool)
        )
        WARM_PROCS[tool] = p
        WARM_STATE[tool]["pid"] = p.pid
        threading.Thread(target=_warm_reader, args=(tool, p), daemon=True).start()

def stop_warm_worker(tool: str):
    with WARM_LOCKS[tool]:
        set_warm_state(tool, "stopping", "Releasing model from GPU...", enabled=False)
        p = WARM_PROCS.get(tool)
        if not p:
            set_warm_state(tool, "off", "Cold mode", enabled=False)
            return
        try:
            p.stdin.write(json.dumps({"op": "stop"}) + "\n")
            p.stdin.flush()
            p.wait(timeout=20)
        except Exception:
            try: p.terminate()
            except Exception: pass
            try: p.wait(timeout=5)
            except Exception:
                try: p.kill()
                except Exception: pass
        WARM_PROCS.pop(tool, None)
        set_warm_state(tool, "off", "Cold mode", enabled=False)

def warm_enabled(tool: str):
    with WARM_LOCKS[tool]:
        p = WARM_PROCS.get(tool)
        return bool(WARM_STATE[tool]["enabled"] and p and p.poll() is None)

def wait_warm_ready(tool: str, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with WARM_LOCKS[tool]:
            st = WARM_STATE[tool]["state"]
            p = WARM_PROCS.get(tool)
            if st == "ready" and p and p.poll() is None:
                return
            if st == "error":
                raise RuntimeError(WARM_STATE[tool]["message"])
        time.sleep(0.1)
    raise TimeoutError(f"Timed out loading {TOOL_META[tool]['name']}")

def submit_warm_job(job, payload, success_message):
    tool = job.tool
    start_warm_worker(tool)
    wait_warm_ready(tool)
    event = threading.Event()
    record = {"job": job, "event": event, "ok": False, "error": None}
    with WARM_LOCKS[tool]:
        if tool in WARM_CURRENT:
            raise RuntimeError(f"A {TOOL_META[tool]['name']} warm job is already running.")
        p = WARM_PROCS.get(tool)
        if not p or p.poll() is not None:
            raise RuntimeError("Warm worker is not running.")
        WARM_CURRENT[tool] = record
        job.status = "running"
        job.started_at = time.time()
        emit(job, "log", f"🔥 MODEL ALREADY WARM — reusing {TOOL_META[tool]['name']} in GPU VRAM.", warm=True)
        emit(job, "status", f"Warm model ready · {TOOL_META[tool]['name']} processing started.", status="running", warm=True)
        p.stdin.write(json.dumps({"op": "job", "job_id": job.id, "args": payload}) + "\n")
        p.stdin.flush()
    def waiter():
        event.wait()
        with WARM_LOCKS[tool]:
            WARM_CURRENT.pop(tool, None)
        if record["ok"]:
            try:
                links = upload_job_outputs(job)
                mark_done(job, "completed", success_message, 0, downloads=links, storage="google_drive", warm=True)
            except Exception as exc:
                mark_done(job, "failed", f"Google Drive upload failed: {exc}")
        else:
            mark_done(job, "failed", record["error"] or f"{TOOL_META[tool]['name']} warm worker failed.")
    threading.Thread(target=waiter, daemon=True).start()

def run_child(job: Job, command, success_message: str):
    with tool_locks[job.tool]:
        try:
            job.status = "running"
            job.started_at = time.time()
            emit(job, "status", f"Starting {TOOL_META[job.tool]['name']}…", status="running")
            emit(job, "command", " ".join(map(str, command)))
            job.process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, universal_newlines=True, env=build_env(job.tool))
            for raw in job.process.stdout or []:
                line = raw.rstrip()
                if line: emit(job, "log", line)
            job.process.wait()
            if job.process.returncode == 0:
                links = upload_job_outputs(job)
                mark_done(job, "completed", success_message, 0, downloads=links, storage="google_drive", warm=False)
            else:
                mark_done(job, "failed", f"{TOOL_META[job.tool]['name']} failed (exit code {job.process.returncode}).", job.process.returncode)
        except Exception as exc:
            mark_done(job, "failed", f"{type(exc).__name__}: {exc}")
        finally:
            try:
                if job.process and job.process.stdout: job.process.stdout.close()
            except Exception: pass
            job.process = None

def run_or_warm(job, command, success_message, payload=None):
    if job.warm:
        submit_warm_job(job, payload or {}, success_message)
    else:
        start_thread(run_child, job, command, success_message)

def start_thread(target, *args):
    threading.Thread(target=target, args=args, daemon=True).start()

def create_job(tool: str, filename: str):
    job = Job(tool, filename)
    with jobs_lock: jobs[job.id] = job
    return job

def tool_busy(tool: str):
    with jobs_lock:
        return any(j.tool == tool and j.status in {"queued", "running"} for j in jobs.values())

async def save_upload(upload: UploadFile, target: Path):
    with open(target, "wb") as f:
        while chunk := await upload.read(8 * 1024 * 1024):
            f.write(chunk)
    await upload.close()

# =============================================================================
# 5. WORKERS
# =============================================================================

def worker_chatterbox(job: Job, mode: str, script: Path, ref: Path | None, refs_dir: Path | None,
                      temperature: float, pause: float, max_chars: int):
    command = [
        sys.executable, "-u", str(CHATTERBOX_RUNNER),
        "--mode", mode,
        "--script", str(script),
        "--output", str(job.output_path),
        "--temperature", str(temperature),
        "--pause", str(pause),
        "--max-chars", str(max_chars),
    ]
    if mode == "single":
        command += ["--ref-audio", str(ref)]
    else:
        if ref and ref.exists():
            try:
                with zipfile.ZipFile(ref, "r") as z:
                    root = refs_dir.resolve()
                    for member in z.infolist():
                        target = (refs_dir / member.filename).resolve()
                        if target != root and not str(target).startswith(str(root) + os.sep):
                            raise RuntimeError(f"Unsafe ZIP entry: {member.filename}")
                    z.extractall(refs_dir)

                audio_files = []
                for p in refs_dir.rglob("*"):
                    if p.is_file() and p.suffix.lower() in CHATTERBOX_AUDIO:
                        audio_files.append(p)
                for p in sorted(audio_files):
                    if p.parent == refs_dir:
                        continue
                    target = refs_dir / p.name
                    if target.exists():
                        target = refs_dir / f"{p.stem}_{uuid.uuid4().hex[:6]}{p.suffix.lower()}"
                    shutil.move(str(p), str(target))
                for d in sorted([p for p in refs_dir.rglob("*") if p.is_dir()], key=lambda x: len(x.parts), reverse=True):
                    try:
                        d.rmdir()
                    except OSError:
                        pass
                emit(job, "log", "[UI] Reference ZIP extracted and normalized.")
                flat_refs = sorted(p.name for p in refs_dir.iterdir() if p.is_file())
                for name in flat_refs:
                    emit(job, "log", f"[UI] Reference: {name}")
            except Exception as exc:
                mark_done(job, "failed", f"Reference ZIP error: {exc}")
                return
        command += ["--refs-dir", str(refs_dir)]
    run_or_warm(job, command, "Chatterbox generation completed.", {"mode": mode, "script": str(script), "output": str(job.output_path), "temperature": float(temperature), "pause": float(pause), "max_chars": int(max_chars), **({"ref_audio": str(ref)} if mode == "single" else {"refs_dir": str(refs_dir)})})


def worker_whisper(job: Job, language: str):
    command = [
        sys.executable, "-u", str(WHISPER_RUNNER),
        "--audio", str(job.input_path),
        "--output-words", str(job.words_path),
        "--output-segments", str(job.segments_path),
        "--output-segments-txt", str(job.segments_txt_path),
        "--output-words-txt", str(job.words_txt_path),
        "--output-segment-srt", str(job.segment_srt_path),
        "--output-word-srt", str(job.word_srt_path),
        "--language", language,
        "--device", "cuda",
    ]
    run_or_warm(job, command, "Whisper transcription completed.", {"audio": str(job.input_path), "output_words": str(job.words_path), "output_segments": str(job.segments_path), "output_segments_txt": str(job.segments_txt_path), "output_words_txt": str(job.words_txt_path), "output_segment_srt": str(job.segment_srt_path), "output_word_srt": str(job.word_srt_path), "language": language, "device": "cuda"})


def worker_upscaler(job: Job, scale: float):
    command = [
        sys.executable, "-u", str(UPSCALER_RUNNER),
        "--input", str(job.input_path),
        "--output", str(job.output_path),
        "--scale", str(scale),
        "--device", "cuda",
        "--model-dir", str(UPSCALER_MODELS),
        "--tile", "512",
        "--tile-pad", "32",
    ]
    run_or_warm(job, command, "Real-ESRGAN upscaling completed.", {"input": str(job.input_path), "output": str(job.output_path), "scale": float(scale)})

# =============================================================================
# 6. FASTAPI (REDESIGNED UI)
# =============================================================================

app = FastAPI(title="AI Flows Hub")

HTML = r'''<!doctype html>
<html lang="en" data-theme="dark">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>AI Flows Hub Dashboard</title>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
:root {
    --bg-main: #f8fafc;
    --bg-card: #ffffff;
    --bg-elevated: #f1f5f9;
    --border: #e2e8f0;
    --border-strong: #cbd5e1;
    --text-main: #0f172a;
    --text-muted: #64748b;
    --accent: #4f46e5;
    --accent-hover: #4338ca;
    --accent-bg: #e0e7ff;
    --accent-fg: #3730a3;
    --danger: #ef4444;
    --danger-bg: #fee2e2;
    --danger-fg: #b91c1c;
    --success: #10b981;
    --success-bg: #d1fae5;
    --success-fg: #047857;
    --warning: #f59e0b;
    --shadow-sm: 0 1px 2px 0 rgb(0 0 0 / 0.05);
    --shadow-md: 0 4px 6px -1px rgb(0 0 0 / 0.1), 0 2px 4px -2px rgb(0 0 0 / 0.1);
    --shadow-lg: 0 10px 15px -3px rgb(0 0 0 / 0.1), 0 4px 6px -4px rgb(0 0 0 / 0.1);
    --radius-sm: 8px;
    --radius-md: 12px;
    --radius-lg: 16px;
    --radius-pill: 9999px;
    --font-sans: 'Inter', system-ui, sans-serif;
    --font-mono: 'JetBrains Mono', monospace;
    transition: background-color 0.3s ease, border-color 0.3s ease;
}

[data-theme="dark"] {
    --bg-main: #09090b;
    --bg-card: #18181b;
    --bg-elevated: #27272a;
    --border: #3f3f46;
    --border-strong: #52525b;
    --text-main: #fafafa;
    --text-muted: #a1a1aa;
    --accent: #6366f1;
    --accent-hover: #818cf8;
    --accent-bg: rgba(99, 102, 241, 0.15);
    --accent-fg: #c7d2fe;
    --danger: #ef4444;
    --danger-bg: rgba(239, 68, 68, 0.15);
    --danger-fg: #fecaca;
    --success: #10b981;
    --success-bg: rgba(16, 185, 129, 0.15);
    --success-fg: #a7f3d0;
    --warning: #fbbf24;
    --shadow-sm: 0 1px 2px 0 rgb(0 0 0 / 0.5);
    --shadow-md: 0 4px 6px -1px rgb(0 0 0 / 0.5);
    --shadow-lg: 0 10px 15px -3px rgb(0 0 0 / 0.5);
}

* { box-sizing: border-box; }
body { margin: 0; background: var(--bg-main); color: var(--text-main); font-family: var(--font-sans); height: 100vh; overflow: hidden; display: flex; flex-direction: column; }
button, input, select { font-family: inherit; }

/* HEADER & NAVBAR */
.app-header { height: 72px; padding: 0 32px; display: flex; align-items: center; justify-content: space-between; background: var(--bg-card); border-bottom: 1px solid var(--border); box-shadow: var(--shadow-sm); flex-shrink: 0; }
.brand { display: flex; align-items: center; gap: 16px; }
.brand-icon { width: 40px; height: 40px; background: var(--accent-bg); color: var(--accent); border-radius: var(--radius-md); display: flex; align-items: center; justify-content: center; font-size: 20px; box-shadow: inset 0 0 0 1px rgba(0,0,0,0.05); }
.brand h1 { margin: 0; font-size: 16px; font-weight: 700; letter-spacing: -0.3px; }
.brand p { margin: 2px 0 0; font-size: 12px; color: var(--text-muted); }
.header-actions { display: flex; align-items: center; gap: 20px; }
.runtime { display: flex; align-items: center; gap: 8px; font-size: 12px; font-weight: 500; color: var(--text-muted); }
.dot { width: 8px; height: 8px; border-radius: 50%; background: var(--success); box-shadow: 0 0 8px var(--success); }
.theme-toggle { background: transparent; border: 1px solid var(--border); border-radius: var(--radius-pill); padding: 6px 12px; cursor: pointer; color: var(--text-main); font-size: 12px; font-weight: 600; display: flex; align-items: center; gap: 8px; transition: 0.2s; }
.theme-toggle:hover { background: var(--bg-elevated); }

/* MAIN LAYOUT */
.main-wrapper { display: flex; flex-direction: column; flex: 1; overflow: hidden; width: 100%; max-width: 1600px; margin: 0 auto; }
.sidebar { width: 100%; padding: 16px 24px; border-bottom: 1px solid var(--border); display: flex; align-items: center; gap: 8px; background: var(--bg-main); flex-shrink: 0; }
.sidebar-title { font-size: 11px; text-transform: uppercase; font-weight: 700; color: var(--text-muted); letter-spacing: 0.5px; margin-right: 8px; padding-left: 12px; }
.tab { width: auto; text-align: left; background: transparent; border: none; padding: 12px 16px; border-radius: var(--radius-md); cursor: pointer; font-size: 14px; font-weight: 600; color: var(--text-muted); display: flex; align-items: center; gap: 12px; transition: all 0.2s ease; }
.tab:hover { background: var(--bg-elevated); color: var(--text-main); }
.tab.active { background: var(--accent); color: #fff; box-shadow: var(--shadow-md); }

.content-area { flex: 1; padding: 24px 32px; overflow-y: auto; min-height: 0; }
.tool { display: none; height: 100%; animation: fadeIn 0.3s ease; }
.tool.active { display: grid; grid-template-columns: minmax(0, 4fr) minmax(0, 6fr); gap: 24px; }
@keyframes fadeIn { from { opacity: 0; transform: translateY(5px); } to { opacity: 1; transform: translateY(0); } }

/* PANELS */
.card { background: var(--bg-card); border: 1px solid var(--border); border-radius: var(--radius-lg); box-shadow: var(--shadow-md); display: flex; flex-direction: column; overflow: hidden; min-width: 0; }
.controls-panel { padding: 24px; overflow-y: auto; min-width: 0; }
.activity-panel { background: #000; border: 1px solid var(--border); border-radius: var(--radius-lg); display: flex; flex-direction: column; overflow: hidden; box-shadow: var(--shadow-md); min-width: 0; }

/* TOOL HEADER */
.tool-header { display: flex; justify-content: space-between; align-items: flex-start; margin-bottom: 24px; padding-bottom: 16px; border-bottom: 1px solid var(--border); }
.tool-title h2 { margin: 0; font-size: 22px; font-weight: 800; letter-spacing: -0.5px; }
.tool-title p { margin: 6px 0 0; font-size: 13px; color: var(--text-muted); }
.tool-meta { display: flex; flex-direction: column; align-items: flex-end; gap: 8px; }

/* BADGES & CONTROLS */
.badge { padding: 4px 10px; border-radius: var(--radius-pill); font-size: 11px; font-weight: 700; border: 1px solid var(--border); background: var(--bg-elevated); color: var(--text-muted); }
.badge.ready { background: var(--success-bg); color: var(--success-fg); border-color: transparent; }
.warmctl { background: var(--bg-elevated); border: 1px solid var(--border); padding: 8px 12px; border-radius: var(--radius-md); font-size: 11px; font-weight: 700; color: var(--text-muted); cursor: pointer; display: flex; align-items: center; gap: 6px; transition: all 0.2s; }
.warmctl:hover { border-color: var(--border-strong); }
.warmctl.on { background: var(--success-bg); color: var(--success-fg); border-color: transparent; }
.warmctl.loading { background: var(--accent-bg); color: var(--accent-fg); border-color: transparent; }
.warmstate { font-size: 11px; color: var(--text-muted); }

/* FORMS & INPUTS */
.section { margin-bottom: 20px; }
.label { font-size: 12px; font-weight: 700; color: var(--text-main); margin-bottom: 10px; display: block; }
.toggle { display: flex; background: var(--bg-elevated); padding: 4px; border-radius: var(--radius-md); gap: 4px; border: 1px solid var(--border); }
.toggle button { flex: 1; padding: 8px; border: none; background: transparent; border-radius: var(--radius-sm); font-size: 12px; font-weight: 600; color: var(--text-muted); cursor: pointer; transition: 0.2s; }
.toggle button.active { background: var(--bg-card); color: var(--text-main); box-shadow: var(--shadow-sm); }

/* DROPZONES */
.drop { border: 2px dashed var(--border-strong); border-radius: var(--radius-md); background: var(--bg-elevated); padding: 24px; text-align: center; display: flex; flex-direction: column; align-items: center; transition: all 0.2s; cursor: pointer; }
.drop:hover, .drop.drag { border-color: var(--accent); background: var(--accent-bg); }
.drop .big { font-size: 32px; margin-bottom: 12px; }
.drop b { font-size: 14px; font-weight: 600; color: var(--text-main); }
.drop span { font-size: 12px; color: var(--text-muted); margin-top: 6px; }
.choose { margin-top: 16px; padding: 8px 16px; background: var(--bg-card); border: 1px solid var(--border); border-radius: var(--radius-pill); font-size: 12px; font-weight: 600; cursor: pointer; color: var(--text-main); box-shadow: var(--shadow-sm); transition: 0.2s; }
.choose:hover { border-color: var(--text-muted); }
.hidden { display: none; }
.file { margin-top: 12px; display: none; background: var(--bg-elevated); border: 1px solid var(--border); padding: 12px 16px; border-radius: var(--radius-md); }
.file strong { font-size: 13px; display: block; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; color: var(--text-main); }
.file span { font-size: 11px; color: var(--text-muted); margin-top: 4px; display: block; }

/* SLIDERS & SELECTS */
.seg3 { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; }
.seg { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }
.setting { background: var(--bg-elevated); border: 1px solid var(--border); padding: 12px; border-radius: var(--radius-md); }
.srow { display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; }
.setting small { font-size: 11px; font-weight: 600; color: var(--text-muted); }
.setting strong { font-size: 12px; font-weight: 700; color: var(--text-main); }
.range { width: 100%; accent-color: var(--accent); cursor: pointer; }
.select, .number { width: 100%; padding: 10px 12px; background: var(--bg-card); border: 1px solid var(--border); border-radius: var(--radius-md); color: var(--text-main); font-size: 13px; font-weight: 500; outline: none; box-shadow: var(--shadow-sm); transition: 0.2s; }
.select:focus, .number:focus { border-color: var(--accent); box-shadow: 0 0 0 2px var(--accent-bg); }

/* METERS */
.meters { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-top: 24px; }
.meter { background: var(--bg-elevated); border: 1px solid var(--border); padding: 12px; border-radius: var(--radius-md); }
.meter small { display: block; font-size: 10px; text-transform: uppercase; font-weight: 700; color: var(--text-muted); letter-spacing: 0.5px; }
.meter b { display: block; margin-top: 6px; font-size: 13px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }

/* MISC */
.hint { padding: 12px 16px; background: var(--bg-elevated); border: 1px solid var(--border); border-radius: var(--radius-md); font-size: 12px; color: var(--text-muted); line-height: 1.5; margin-bottom: 24px; }
.hint b { color: var(--text-main); }
.modebox { display: none; }
.modebox.active { display: block; }

/* BUTTONS */
.actions { display: grid; grid-template-columns: 2fr 1fr; gap: 12px; margin-top: 24px; }
.btn { padding: 14px; border: none; border-radius: var(--radius-md); font-size: 14px; font-weight: 700; cursor: pointer; transition: 0.2s; text-align: center; }
.primary { background: var(--accent); color: #fff; box-shadow: var(--shadow-sm); }
.primary:hover:not(:disabled) { background: var(--accent-hover); box-shadow: var(--shadow-md); }
.danger { background: transparent; border: 1px solid var(--danger); color: var(--danger); }
.danger:hover:not(:disabled) { background: var(--danger-bg); }
.primary:disabled, .danger:disabled { opacity: 0.5; cursor: not-allowed; }

/* PROGRESS & STATUS */
.progress { display: none; margin-top: 20px; background: var(--bg-elevated); padding: 16px; border-radius: var(--radius-md); border: 1px solid var(--border); }
.prow { display: flex; justify-content: space-between; font-size: 11px; font-weight: 600; color: var(--text-muted); margin-bottom: 8px; }
.track { height: 6px; background: var(--border-strong); border-radius: var(--radius-pill); overflow: hidden; margin-bottom: 12px; }
.track:last-child { margin-bottom: 0; }
.bar { height: 100%; width: 0%; background: var(--accent); border-radius: inherit; transition: width 0.2s ease; }
.status { margin-top: 20px; padding: 12px 16px; border-radius: var(--radius-md); font-size: 12px; font-weight: 600; border: 1px solid var(--border); background: var(--bg-elevated); color: var(--text-muted); display: flex; align-items: center; gap: 8px; }
.status.ok { background: var(--success-bg); border-color: transparent; color: var(--success-fg); }
.status.err { background: var(--danger-bg); border-color: transparent; color: var(--danger-fg); }

/* OUTPUT & DOWNLOADS */
.output { display: none; margin-top: 24px; padding: 20px; background: var(--success-bg); border: 1px solid rgba(16, 185, 129, 0.3); border-radius: var(--radius-md); }
.output b { font-size: 14px; color: var(--success-fg); display: block; margin-bottom: 12px; }
.downloads { display: flex; flex-wrap: wrap; gap: 8px; }
.download { display: inline-flex; align-items: center; padding: 8px 14px; background: #fff; color: #000; border-radius: var(--radius-sm); font-size: 12px; font-weight: 700; text-decoration: none; box-shadow: var(--shadow-sm); transition: 0.2s; }
.download:hover { transform: translateY(-1px); box-shadow: var(--shadow-md); }
.download.alt { background: var(--bg-card); color: var(--text-main); border: 1px solid var(--border); }
[data-theme="dark"] .download { background: var(--accent); color: #fff; border: none; }
[data-theme="dark"] .download.alt { background: var(--bg-elevated); color: var(--text-main); border: 1px solid var(--border); }

/* LOGS (TERMINAL) */
.activity-head { padding: 16px 20px; border-bottom: 1px solid #333; display: flex; justify-content: space-between; align-items: center; font-size: 12px; font-weight: 700; color: #a1a1aa; background: #09090b; }
.logs { flex: 1; padding: 20px; overflow-y: auto; font-family: var(--font-mono); font-size: 11px; line-height: 1.6; color: #d4d4d8; background: #000; }
.log { margin-bottom: 6px; word-break: break-word; white-space: pre-wrap; }
.log.dim { color: #52525b; }
.log.ok { color: #34d399; }
.log.err { color: #f87171; }
.log.info { color: #60a5fa; }

/* FOOTER */
.footer { position: fixed; bottom: 16px; right: 32px; font-size: 11px; color: var(--text-muted); font-weight: 500; pointer-events: none; }

@media(max-width: 1024px) {
    .tool.active { grid-template-columns: minmax(0, 4fr) minmax(0, 6fr); }
    .activity-panel { height: auto; }
}
@media(max-width: 768px) {
    .main-wrapper { overflow-y: auto; }
    body { overflow: auto; }
    .sidebar { border-bottom: 1px solid var(--border); flex-direction: row; flex-wrap: wrap; padding: 16px; }
    .sidebar-title { display: none; }
    .tab { width: auto; padding: 8px 16px; }
    .content-area { padding: 16px; }
    .tool.active { grid-template-columns: minmax(0, 4fr) minmax(0, 6fr); }
    .activity-panel { min-height: 400px; }
}
@media(max-width: 560px) {
    .seg3, .seg, .meters { grid-template-columns: 1fr; }
    .header-actions { display: none; }
}
</style>
</head>
<body>

<div class="app-header">
    <div class="brand">
        <div class="brand-icon">⚡</div>
        <div>
            <h1>AI Flows Hub</h1>
            <p>Chatterbox · Whisper · Real‑ESRGAN</p>
        </div>
    </div>
    <div class="header-actions">
        <div class="runtime">
            <span class="dot"></span>
            <span id="runtimeText">Checking runners…</span>
        </div>
        <button class="theme-toggle" onclick="toggleTheme()">
            <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><circle cx="12" cy="12" r="5"></circle><line x1="12" y1="1" x2="12" y2="3"></line><line x1="12" y1="21" x2="12" y2="23"></line><line x1="4.22" y1="4.22" x2="5.64" y2="5.64"></line><line x1="18.36" y1="18.36" x2="19.78" y2="19.78"></line><line x1="1" y1="12" x2="3" y2="12"></line><line x1="21" y1="12" x2="23" y2="12"></line><line x1="4.22" y1="19.78" x2="5.64" y2="18.36"></line><line x1="18.36" y1="5.64" x2="19.78" y2="4.22"></line></svg>
            Theme
        </button>
    </div>
</div>

<div class="main-wrapper">
    <aside class="sidebar">
        <div class="sidebar-title">Tools</div>
        <button class="tab active" data-tab="cb">🔊 Chatterbox Turbo</button>
        <button class="tab" data-tab="w">🎙️ Whisper</button>
        <button class="tab" data-tab="u">✨ Real‑ESRGAN</button>
    </aside>

    <main class="content-area">
        <!-- CHATTERBOX -->
        <section id="tool-cb" class="tool active">
            <div class="card controls-panel">
                <div class="tool-header">
                    <div class="tool-title">
                        <h2>Chatterbox Turbo</h2>
                        <p>Generate professional AI voiceovers.</p>
                    </div>
                    <div class="tool-meta">
                        <button id="cbWarm" class="warmctl" type="button">🔥 Warm GPU: OFF</button>
                        <span id="cbBadge" class="badge">Checking…</span>
                        <span id="cbWarmState" class="warmstate">Cold mode</span>
                    </div>
                </div>

                <div class="section">
                    <span class="label">Generation Mode</span>
                    <div class="toggle">
                        <button id="cbSingle" class="active">Single Speaker</button>
                        <button id="cbMulti">Multi Speaker</button>
                    </div>
                </div>

                <div class="section">
                    <span class="label">Script</span>
                    <div id="cbScriptDrop" class="drop">
                        <div class="big">📝</div>
                        <b>Drop your .txt script</b>
                        <span>or choose a file below</span>
                        <label class="choose" for="cbScript">Browse Files</label>
                        <input id="cbScript" class="hidden" type="file" accept=".txt">
                    </div>
                    <div id="cbScriptInfo" class="file">
                        <strong id="cbScriptName"></strong>
                        <span id="cbScriptMeta"></span>
                    </div>
                </div>

                <div class="section modebox active" id="cbSingleBox">
                    <span class="label">Reference Voice</span>
                    <div id="cbRefDrop" class="drop">
                        <div class="big">🎵</div>
                        <b>Drop reference audio</b>
                        <span>WAV, MP3, M4A, FLAC, OGG</span>
                        <label class="choose" for="cbRef">Browse Files</label>
                        <input id="cbRef" class="hidden" type="file" accept=".wav,.mp3,.m4a,.flac,.ogg">
                    </div>
                    <div id="cbRefInfo" class="file">
                        <strong id="cbRefName"></strong>
                        <span id="cbRefMeta"></span>
                    </div>
                </div>

                <div class="section modebox" id="cbMultiBox">
                    <span class="label">Speaker References</span>
                    <div id="cbZipDrop" class="drop">
                        <div class="big">📦</div>
                        <b>Drop reference ZIP</b>
                        <span>speaker_1.wav, speaker_2.wav ...</span>
                        <label class="choose" for="cbZip">Browse Files</label>
                        <input id="cbZip" class="hidden" type="file" accept=".zip">
                    </div>
                    <div id="cbZipInfo" class="file">
                        <strong id="cbZipName"></strong>
                        <span id="cbZipMeta"></span>
                    </div>
                </div>

                <div class="section">
                    <span class="label">Configuration</span>
                    <div class="seg3">
                        <div class="setting">
                            <div class="srow">
                                <small>Temperature</small>
                                <strong id="cbTempVal">1.0</strong>
                            </div>
                            <input id="cbTemp" class="range" type="range" min="0.1" max="2" step="0.1" value="1">
                        </div>
                        <div class="setting">
                            <div class="srow">
                                <small>Pause</small>
                                <strong id="cbPauseVal">0.5s</strong>
                            </div>
                            <input id="cbPause" class="range" type="range" min="0" max="2" step="0.05" value="0.5">
                        </div>
                        <div>
                            <span class="label" style="font-size:11px; color:var(--text-muted); margin-bottom:8px">Max Chars</span>
                            <input id="cbMax" class="number" type="number" min="50" max="500" step="10" value="200">
                        </div>
                    </div>
                </div>

                <div class="hint">
                    <b>GPU isolation:</b> generation runs inside the existing <code>run_tts.py</code> child process. After generation, the WAV is uploaded to Google Drive. Download buttons point directly to Drive.
                </div>

                <div class="actions">
                    <button id="cbStart" class="btn primary" disabled>Generate Voice</button>
                    <button id="cbCancel" class="btn danger" disabled>Cancel</button>
                </div>

                <div id="cbProgress" class="progress">
                    <div class="prow">
                        <span>Upload Progress</span>
                        <span id="cbUploadPct">0%</span>
                    </div>
                    <div class="track"><div id="cbUploadBar" class="bar"></div></div>
                    <div class="prow" style="margin-top:16px">
                        <span>Processing Progress</span>
                        <span id="cbProcPct">0%</span>
                    </div>
                    <div class="track"><div id="cbProcBar" class="bar"></div></div>
                </div>

                <div id="cbStatus" class="status">Ready when you are.</div>

                <div class="meters">
                    <div class="meter"><small>Phase</small><b id="cbPhase">—</b></div>
                    <div class="meter"><small>Progress</small><b id="cbChunk">—</b></div>
                    <div class="meter"><small>Mode</small><b id="cbModeStat">Single</b></div>
                    <div class="meter"><small>Temp</small><b id="cbTempStat">1.0</b></div>
                </div>

                <div id="cbOutput" class="output">
                    <b>✅ Voice generation successful</b>
                    <div class="downloads">
                        <a id="cbDownload" class="download" href="#">Download WAV</a>
                    </div>
                </div>
            </div>

            <div class="activity-panel">
                <div class="activity-head">
                    <span>Terminal Activity</span>
                    <span class="badge" style="background: transparent; color: #a1a1aa; border-color: #3f3f46">DRIVE UPLOAD</span>
                </div>
                <div id="cbLogs" class="logs">
                    <div class="log dim">No Chatterbox job yet.</div>
                </div>
            </div>
        </section>

        <!-- WHISPER -->
        <section id="tool-w" class="tool">
            <div class="card controls-panel">
                <div class="tool-header">
                    <div class="tool-title">
                        <h2>Whisper</h2>
                        <p>Transcribe audio with word-level output.</p>
                    </div>
                    <div class="tool-meta">
                        <button id="wWarm" class="warmctl" type="button">🔥 Warm GPU: OFF</button>
                        <span id="wBadge" class="badge">Checking…</span>
                        <span id="wWarmState" class="warmstate">Cold mode</span>
                    </div>
                </div>

                <div class="section">
                    <span class="label">Audio File</span>
                    <div id="wDrop" class="drop">
                        <div class="big">🎙️</div>
                        <b>Drop your audio file</b>
                        <span>WAV, MP3, M4A, FLAC, OGG, AAC, WMA</span>
                        <label class="choose" for="wFile">Browse Files</label>
                        <input id="wFile" class="hidden" type="file" accept=".wav,.mp3,.m4a,.flac,.ogg,.aac,.wma">
                    </div>
                    <div id="wInfo" class="file">
                        <strong id="wName"></strong>
                        <span id="wMeta"></span>
                    </div>
                </div>

                <div class="seg">
                    <div>
                        <span class="label">Language</span>
                        <select id="wLang" class="select">
                            <option value="auto">Auto-detect</option>
                            <option value="en">English</option>
                            <option value="ur">Urdu</option>
                            <option value="hi">Hindi</option>
                            <option value="ar">Arabic</option>
                            <option value="es">Spanish</option>
                            <option value="fr">French</option>
                            <option value="de">German</option>
                            <option value="it">Italian</option>
                            <option value="pt">Portuguese</option>
                            <option value="ru">Russian</option>
                            <option value="zh">Chinese</option>
                            <option value="ja">Japanese</option>
                            <option value="ko">Korean</option>
                            <option value="tr">Turkish</option>
                        </select>
                    </div>
                    <div>
                        <span class="label">Hardware</span>
                        <div class="setting" style="height:44px; display:flex; align-items:center; justify-content: space-between">
                            <strong style="color:var(--success); font-size:13px">CUDA</strong>
                            <small>existing runner</small>
                        </div>
                    </div>
                </div>

                <div class="hint" style="margin-top:24px">
                    <b>Output:</b> Whisper generates six formats and uploads them directly to Google Drive. Download buttons use the Google Drive URL.
                </div>

                <div class="actions">
                    <button id="wStart" class="btn primary" disabled>Start Transcription</button>
                    <button id="wCancel" class="btn danger" disabled>Cancel</button>
                </div>

                <div id="wProgress" class="progress">
                    <div class="prow">
                        <span>Upload Progress</span>
                        <span id="wUploadPct">0%</span>
                    </div>
                    <div class="track"><div id="wUploadBar" class="bar"></div></div>
                    <div class="prow" style="margin-top:16px">
                        <span>Processing Progress</span>
                        <span id="wProcPct">0%</span>
                    </div>
                    <div class="track"><div id="wProcBar" class="bar"></div></div>
                </div>

                <div id="wStatus" class="status">Ready when you are.</div>

                <div class="meters">
                    <div class="meter"><small>Phase</small><b id="wPhase">—</b></div>
                    <div class="meter"><small>Language</small><b id="wDetected">—</b></div>
                    <div class="meter"><small>Segments</small><b id="wSeg">—</b></div>
                    <div class="meter"><small>Words</small><b id="wWords">—</b></div>
                </div>

                <div id="wOutput" class="output">
                    <b>✅ Transcription successful</b>
                    <div class="downloads">
                        <a id="wWordsDl" class="download" href="#">📄 Words JSON</a>
                        <a id="wSegDl" class="download alt" href="#">📄 Segment JSON</a>
                        <a id="wSegTxtDl" class="download alt" href="#">📄 Segment TXT</a>
                        <a id="wWordsTxtDl" class="download alt" href="#">📄 Words TXT</a>
                        <a id="wSegSrtDl" class="download alt" href="#">📄 Segment SRT</a>
                        <a id="wWordSrtDl" class="download alt" href="#">📄 Word SRT</a>
                    </div>
                </div>
            </div>

            <div class="activity-panel">
                <div class="activity-head">
                    <span>Terminal Activity</span>
                    <span class="badge" style="background: transparent; color: #a1a1aa; border-color: #3f3f46">DRIVE UPLOAD</span>
                </div>
                <div id="wLogs" class="logs">
                    <div class="log dim">No Whisper job yet.</div>
                </div>
            </div>
        </section>

        <!-- UPSCALER -->
        <section id="tool-u" class="tool">
            <div class="card controls-panel">
                <div class="tool-header">
                    <div class="tool-title">
                        <h2>Real-ESRGAN</h2>
                        <p>Upscale one image or a ZIP batch.</p>
                    </div>
                    <div class="tool-meta">
                        <button id="uWarm" class="warmctl" type="button">🔥 Warm GPU: OFF</button>
                        <span id="uBadge" class="badge">Checking…</span>
                        <span id="uWarmState" class="warmstate">Cold mode</span>
                    </div>
                </div>

                <div class="section">
                    <span class="label">Input Source</span>
                    <div id="uDrop" class="drop">
                        <div class="big">🖼️</div>
                        <b>Drop image or ZIP batch</b>
                        <span>PNG, JPG, JPEG, WEBP, BMP, TIF, TIFF, ZIP</span>
                        <label class="choose" for="uFile">Browse Files</label>
                        <input id="uFile" class="hidden" type="file" accept=".png,.jpg,.jpeg,.webp,.bmp,.tif,.tiff,.zip">
                    </div>
                    <div id="uInfo" class="file">
                        <strong id="uName"></strong>
                        <span id="uMeta"></span>
                    </div>
                </div>

                <div class="seg">
                    <div>
                        <span class="label">Scale Factor</span>
                        <select id="uScale" class="select">
                            <option value="2">2×</option>
                            <option value="3.5">3.5×</option>
                            <option value="4" selected>4×</option>
                        </select>
                    </div>
                    <div>
                        <span class="label">Hardware</span>
                        <div class="setting" style="height:44px; display:flex; align-items:center; justify-content: space-between">
                            <strong style="color:var(--success); font-size:13px">CUDA</strong>
                            <small>Tile 512</small>
                        </div>
                    </div>
                </div>

                <div class="hint" style="margin-top:24px">
                    <b>Batch friendly:</b> ZIP uploads are passed to the existing Real-ESRGAN runner. The completed file is uploaded to Google Drive for direct download.
                </div>

                <div class="actions">
                    <button id="uStart" class="btn primary" disabled>Start Upscaling</button>
                    <button id="uCancel" class="btn danger" disabled>Cancel</button>
                </div>

                <div id="uProgress" class="progress">
                    <div class="prow">
                        <span>Upload Progress</span>
                        <span id="uUploadPct">0%</span>
                    </div>
                    <div class="track"><div id="uUploadBar" class="bar"></div></div>
                    <div class="prow" style="margin-top:16px">
                        <span>Processing Progress</span>
                        <span id="uProcPct">0%</span>
                    </div>
                    <div class="track"><div id="uProcBar" class="bar"></div></div>
                </div>

                <div id="uStatus" class="status">Ready when you are.</div>

                <div class="meters">
                    <div class="meter"><small>Phase</small><b id="uPhase">—</b></div>
                    <div class="meter"><small>Progress</small><b id="uProgressTxt">—</b></div>
                    <div class="meter"><small>Scale</small><b id="uScaleStat">4×</b></div>
                    <div class="meter"><small>Input</small><b id="uType">—</b></div>
                </div>

                <div id="uOutput" class="output">
                    <b>✅ Upscaling successful</b>
                    <div class="downloads">
                        <a id="uDownload" class="download" href="#">Download Output</a>
                    </div>
                </div>
            </div>

            <div class="activity-panel">
                <div class="activity-head">
                    <span>Terminal Activity</span>
                    <span class="badge" style="background: transparent; color: #a1a1aa; border-color: #3f3f46">DRIVE UPLOAD</span>
                </div>
                <div id="uLogs" class="logs">
                    <div class="log dim">No Real-ESRGAN job yet.</div>
                </div>
            </div>
        </section>
    </main>
</div>

<div class="footer">Three tools · Isolated execution · Drive Integration</div>

<script>
// THEME TOGGLE LOGIC
function toggleTheme() {
    const html = document.documentElement;
    const current = html.getAttribute('data-theme');
    html.setAttribute('data-theme', current === 'dark' ? 'light' : 'dark');
}

// ORIGINAL JS LOGIC
const $=id=>document.getElementById(id);
const CHAT_AUDIO=new Set(['.wav','.mp3','.m4a','.flac','.ogg']);
const WH_AUDIO=new Set(['.wav','.mp3','.m4a','.flac','.ogg','.aac','.wma']);
const U_EXTS=new Set(['.png','.jpg','.jpeg','.webp','.bmp','.tif','.tiff','.zip']);
const S={cb:{script:null,ref:null,zip:null,mode:'single',job:null,ws:null,warm:false},w:{file:null,job:null,ws:null,warm:false},u:{file:null,job:null,ws:null,warm:false}};
const toolName={cb:'Chatterbox',w:'Whisper',u:'Real-ESRGAN'};
const boxes={cb:'cbLogs',w:'wLogs',u:'uLogs'};

function addLog(k,msg,cls=''){const b=$(boxes[k]);const d=document.createElement('div');d.className='log '+cls;d.textContent=msg;b.appendChild(d);b.scrollTop=b.scrollHeight;}
function resetLogs(k,msg){$(boxes[k]).innerHTML='';addLog(k,msg,'dim')}
function setStatus(k,msg,ok=false,err=false){const id=k==='cb'?'cbStatus':k==='w'?'wStatus':'uStatus';$(id).className='status'+(ok?' ok':'')+(err?' err':'');$(id).textContent=msg}
function size(n){return n<1024*1024?(n/1024).toFixed(1)+' KB':(n/1024/1024).toFixed(2)+' MB'}
function fileInfo(info,name,meta,f){$(info).style.display='block';$(name).textContent=f.name;$(meta).textContent=size(f.size)}
function setupDrop(drop,input,cb){const el=$(drop), inp=$(input); inp.addEventListener("click",e=>{e.stopPropagation();inp.value='';}); el.addEventListener('dragover',e=>{e.preventDefault();el.classList.add('drag')});['dragleave','drop'].forEach(v=>el.addEventListener(v,e=>{e.preventDefault();el.classList.remove('drag')}));el.addEventListener('drop',e=>{const f=e.dataTransfer.files[0];if(f)cb(f)});inp.addEventListener('change',()=>{if(inp.files[0])cb(inp.files[0])})}
function ext(n){return '.'+n.split('.').pop().toLowerCase()}
function upload(url,form,progress,done){const x=new XMLHttpRequest();x.open('POST',url);x.upload.onprogress=e=>{if(e.lengthComputable)progress(e.loaded/e.total*100)};x.onerror=()=>done(null,{error:'Network/upload error.'},0);x.onload=()=>{let d={};try{d=JSON.parse(x.responseText)}catch{d={error:'Invalid server response.'}}done(x,d,x.status)};x.send(form)}
function connect(k,id){const protocol=location.protocol==='https:'?'wss':'ws';const ws=new WebSocket(protocol+'://'+location.host+'/ws/'+id);S[k].ws=ws;ws.onmessage=e=>event(k,JSON.parse(e.data));ws.onerror=()=>addLog(k,'⚠ WebSocket error.','err');ws.onclose=()=>{S[k].ws=null}}

function cbParse(line){let m=line.match(/Loading ChatterboxTurboTTS/i);if(m){$('cbPhase').textContent='Loading model';$('cbProcBar').style.width='4%';$('cbProcPct').textContent='4%'}if(line.includes('Model loaded')){$('cbPhase').textContent='Model ready';$('cbProcBar').style.width='10%';$('cbProcPct').textContent='10%'}m=line.match(/Script split into\s+(\d+)\s+chunk/);if(m){$('cbChunk').textContent='0/'+m[1];$('cbPhase').textContent='Generating';$('cbProcBar').style.width='15%';$('cbProcPct').textContent='15%'}m=line.match(/(?:Chunk|chunk)\s+(\d+)\s*\/\s*(\d+)/);if(m){const p=Number(m[1])/Number(m[2])*100;$('cbChunk').textContent=m[1]+'/'+m[2];$('cbPhase').textContent='Generating';$('cbProcBar').style.width=Math.max(15,p)+'%';$('cbProcPct').textContent=Math.max(15,p).toFixed(1)+'%'}if(line.includes('Saved:')){$('cbPhase').textContent='Saving'}if(line.includes('✓ DONE')||line.includes('BATCH COMPLETE')){$('cbProcBar').style.width='100%';$('cbProcPct').textContent='100%';$('cbPhase').textContent='Complete'}}
function wParse(line){if(/Loading model|Creating model/i.test(line)){$('wPhase').textContent='Loading model';$('wProcBar').style.width='3%';$('wProcPct').textContent='3%'}if(line.includes('Model loaded')){$('wPhase').textContent='Transcribing';$('wProcBar').style.width='8%';$('wProcPct').textContent='8%'}let m=line.match(/Detected language:\s+(\w+)\s+\(prob=([\d.]+)\)/);if(m)$('wDetected').textContent=m[1]+' '+m[2];m=line.match(/Segments:\s+(\d+)\s+\|\s+Words:\s+(\d+)/);if(m){$('wSeg').textContent=m[1];$('wWords').textContent=m[2];$('wProcBar').style.width='94%';$('wProcPct').textContent='94%';$('wPhase').textContent='Writing JSON'}if(/Words JSON|Segments JSON/.test(line)){$('wProcBar').style.width='100%';$('wProcPct').textContent='100%';$('wPhase').textContent='Complete'}}
function uParse(line){let m=line.match(/(?:Overall progress|Image progress|Processed|Tile progress|Tile)\s*:?\s*(\d+)\s*\/\s*(\d+)/)||line.match(/\[ZIP\]\s+(?:Extracted|Packaged)\s+(\d+)\s*\/\s*(\d+)/);if(m){const p=Number(m[1])/Number(m[2])*100;$('uProgressTxt').textContent=m[1]+'/'+m[2];$('uProcBar').style.width=p+'%';$('uProcPct').textContent=p.toFixed(1)+'%'}if(/Loading model|Creating model/.test(line)){$('uPhase').textContent='Loading model'}if(line.includes('Saving output')){$('uPhase').textContent='Saving output'}if(line.includes('✓ DONE')||line.includes('BATCH COMPLETE')){$('uProcBar').style.width='100%';$('uProcPct').textContent='100%';$('uPhase').textContent='Complete'}}

function event(k,d){if(d.type==='log'){if(k==='cb')cbParse(d.message);if(k==='w')wParse(d.message);if(k==='u')uParse(d.message);let cls=d.message.includes('✓')?'ok':(/ERROR|failed|✗/i.test(d.message)?'err':/[Whisper]/.test(d.message)?'info':'');addLog(k,d.message,cls);return}if(d.type==='command'){addLog(k,'▶ '+d.message,'dim');return}if(d.type==='status'){if(d.status==='completed'){complete(k,d)}if(d.status==='failed'){fail(k,d.message)}}}
function complete(k,d){S[k].job=null;const links=d?.downloads||{};if(k==='cb'){$('cbProcBar').style.width='100%';$('cbProcPct').textContent='100%';$('cbPhase').textContent='Complete · Drive';$('cbOutput').style.display='block';$('cbDownload').href=links.audio||'#';$('cbDownload').target='_blank';$('cbStart').disabled=false;$('cbCancel').disabled=true;setStatus(k,'Generation completed · output uploaded to Google Drive.',true)}if(k==='w'){$('wProcBar').style.width='100%';$('wProcPct').textContent='100%';$('wPhase').textContent='Complete · Drive';$('wOutput').style.display='block';$('wWordsDl').href=links.words||'#';$('wSegDl').href=links.segments||'#';$('wSegTxtDl').href=links['segments-txt']||'#';$('wWordsTxtDl').href=links['words-txt']||'#';$('wSegSrtDl').href=links['segment-srt']||'#';$('wWordSrtDl').href=links['word-srt']||'#';document.querySelectorAll('#wOutput a.download').forEach(a=>a.target='_blank');$('wStart').disabled=false;$('wCancel').disabled=true;setStatus(k,'Transcription completed · outputs uploaded to Google Drive.',true)}if(k==='u'){$('uProcBar').style.width='100%';$('uProcPct').textContent='100%';$('uPhase').textContent='Complete · Drive';$('uOutput').style.display='block';$('uDownload').href=links.output||'#';$('uDownload').target='_blank';$('uStart').disabled=false;$('uCancel').disabled=true;setStatus(k,'Upscaling completed · output uploaded to Google Drive.',true)}}
function fail(k,msg){S[k].job=null;if(k==='cb'){$('cbStart').disabled=false;$('cbCancel').disabled=true}if(k==='w'){$('wStart').disabled=false;$('wCancel').disabled=true}if(k==='u'){$('uStart').disabled=false;$('uCancel').disabled=true}setStatus(k,'❌ '+msg,false,true)}
function cancel(k){if(!S[k].job)return;fetch('/cancel/'+S[k].job,{method:'POST'}).then(()=>{fail(k,'Job cancelled.')})}

function cbReady(){const ready=!!S.cb.script&&(S.cb.mode==='single'?!!S.cb.ref:!!S.cb.zip);$('cbStart').disabled=!ready}
setupDrop('cbScriptDrop','cbScript',f=>{if(ext(f.name)!=='.txt')return setStatus('cb','Script must be .txt',false,true);S.cb.script=f;fileInfo('cbScriptInfo','cbScriptName','cbScriptMeta',f);cbReady();setStatus('cb','Script ready.');});
setupDrop('cbRefDrop','cbRef',f=>{if(!CHAT_AUDIO.has(ext(f.name)))return setStatus('cb','Unsupported reference audio.',false,true);S.cb.ref=f;fileInfo('cbRefInfo','cbRefName','cbRefMeta',f);cbReady();setStatus('cb','Reference ready.');});
setupDrop('cbZipDrop','cbZip',f=>{if(ext(f.name)!=='.zip')return setStatus('cb','Reference file must be ZIP.',false,true);S.cb.zip=f;fileInfo('cbZipInfo','cbZipName','cbZipMeta',f);cbReady();setStatus('cb','Reference ZIP ready.');});
$('cbSingle').onclick=()=>{S.cb.mode='single';$('cbSingle').classList.add('active');$('cbMulti').classList.remove('active');$('cbSingleBox').classList.add('active');$('cbMultiBox').classList.remove('active');$('cbModeStat').textContent='Single';cbReady()};
$('cbMulti').onclick=()=>{S.cb.mode='multi';$('cbMulti').classList.add('active');$('cbSingle').classList.remove('active');$('cbSingleBox').classList.remove('active');$('cbMultiBox').classList.add('active');$('cbModeStat').textContent='Multi';cbReady()};
$('cbTemp').oninput=e=>{$('cbTempVal').textContent=e.target.value;$('cbTempStat').textContent=e.target.value};$('cbPause').oninput=e=>$('cbPauseVal').textContent=e.target.value+'s';
$('cbStart').onclick=()=>{if(!S.cb.script)return;$('cbStart').disabled=true;$('cbCancel').disabled=false;$('cbOutput').style.display='none';$('cbProgress').style.display='block';$('cbUploadBar').style.width='0%';$('cbUploadPct').textContent='0%';$('cbProcBar').style.width='0%';$('cbProcPct').textContent='0%';$('cbPhase').textContent='Uploading';resetLogs('cb','Starting Chatterbox…');setStatus('cb','Uploading files…');const f=new FormData();f.append('mode',S.cb.mode);f.append('script',S.cb.script);if(S.cb.mode==='single')f.append('ref_audio',S.cb.ref);else f.append('ref_zip',S.cb.zip);f.append('temperature',$('cbTemp').value);f.append('pause',$('cbPause').value);f.append('max_chars',$('cbMax').value);f.append('warm',S.cb.warm?'1':'0');upload('/chatterbox/start',f,p=>{$('cbUploadBar').style.width=p+'%';$('cbUploadPct').textContent=p.toFixed(1)+'%'},(x,d,s)=>{if(s!==200){return fail('cb',d.error||'Upload failed.')}S.cb.job=d.job_id;$('cbUploadBar').style.width='100%';$('cbUploadPct').textContent='100%';$('cbPhase').textContent='Starting';setStatus('cb',S.cb.warm?'Files uploaded · warm GPU worker processing.':'Files uploaded · GPU subprocess starting.');connect('cb',S.cb.job)})};
$('cbCancel').onclick=()=>cancel('cb');

setupDrop('wDrop','wFile',f=>{if(!WH_AUDIO.has(ext(f.name)))return setStatus('w','Unsupported audio format.',false,true);S.w.file=f;fileInfo('wInfo','wName','wMeta',f);$('wStart').disabled=false;$('wOutput').style.display='none';setStatus('w','Audio ready.');resetLogs('w','Audio selected. Ready to transcribe.')});
$('wStart').onclick=()=>{if(!S.w.file)return;$('wStart').disabled=true;$('wCancel').disabled=false;$('wOutput').style.display='none';$('wProgress').style.display='block';$('wUploadBar').style.width='0%';$('wUploadPct').textContent='0%';$('wProcBar').style.width='0%';$('wProcPct').textContent='0%';$('wPhase').textContent='Uploading';resetLogs('w','Starting Whisper…');setStatus('w','Uploading audio…');const f=new FormData();f.append('file',S.w.file);f.append('language',$('wLang').value);f.append('warm',S.w.warm?'1':'0');upload('/whisper/start',f,p=>{$('wUploadBar').style.width=p+'%';$('wUploadPct').textContent=p.toFixed(1)+'%'},(x,d,s)=>{if(s!==200)return fail('w',d.error||'Upload failed.');S.w.job=d.job_id;$('wUploadBar').style.width='100%';$('wUploadPct').textContent='100%';$('wPhase').textContent='Starting';setStatus('w',S.w.warm?'Audio uploaded · warm GPU worker processing.':'Audio uploaded · GPU subprocess starting.');connect('w',S.w.job)})};
$('wCancel').onclick=()=>cancel('w');

setupDrop('uDrop','uFile',f=>{if(!U_EXTS.has(ext(f.name)))return setStatus('u','Unsupported image/ZIP format.',false,true);S.u.file=f;fileInfo('uInfo','uName','uMeta',f);$('uStart').disabled=false;$('uOutput').style.display='none';$('uType').textContent=ext(f.name)==='.zip'?'ZIP batch':'Image';setStatus('u','Input ready.');resetLogs('u','Input selected. Ready to upscale.')});
$('uScale').onchange=()=>{$('uScaleStat').textContent=$('uScale').value+'×'};
$('uStart').onclick=()=>{if(!S.u.file)return;$('uStart').disabled=true;$('uCancel').disabled=false;$('uOutput').style.display='none';$('uProgress').style.display='block';$('uUploadBar').style.width='0%';$('uUploadPct').textContent='0%';$('uProcBar').style.width='0%';$('uProcPct').textContent='0%';$('uPhase').textContent='Uploading';resetLogs('u','Starting Real-ESRGAN…');setStatus('u','Uploading input…');const f=new FormData();f.append('file',S.u.file);f.append('scale',$('uScale').value);f.append('warm',S.u.warm?'1':'0');upload('/upscaler/start',f,p=>{$('uUploadBar').style.width=p+'%';$('uUploadPct').textContent=p.toFixed(1)+'%'},(x,d,s)=>{if(s!==200)return fail('u',d.error||'Upload failed.');S.u.job=d.job_id;$('uUploadBar').style.width='100%';$('uUploadPct').textContent='100%';$('uPhase').textContent='Starting';setStatus('u',S.u.warm?'Input uploaded · warm GPU worker processing.':'Input uploaded · GPU subprocess starting.');connect('u',S.u.job)})};
$('uCancel').onclick=()=>cancel('u');

function activate(tab){document.querySelectorAll('.tab').forEach(x=>x.classList.toggle('active',x.dataset.tab===tab));document.querySelectorAll('.tool').forEach(x=>x.classList.toggle('active',x.id==='tool-'+tab));}
document.querySelectorAll('.tab').forEach(b=>b.onclick=()=>activate(b.dataset.tab));
const warmTools={cb:'chatterbox',w:'whisper',u:'upscaler'};
const warmUi={cb:['cbWarm','cbWarmState'],w:['wWarm','wWarmState'],u:['uWarm','uWarmState']};
function renderWarm(k,c){const wasReady=S[k].warm&&S[k]._warmReadyLogged;S[k].warm=!!c.enabled;const [bid,sid]=warmUi[k],b=$(bid),t=$(sid);b.disabled=!!S[k].job||c.state==='loading'||c.state==='stopping';b.classList.toggle('on',c.enabled&&c.state==='ready');b.classList.toggle('loading',c.enabled&&c.state==='loading');b.textContent=!c.enabled?'🔥 Warm GPU: OFF':c.state==='ready'?'🔥 Warm GPU: ON':'◐ Loading GPU...';t.textContent=c.message||c.state;if(c.state==='ready'&&c.enabled&&!S[k]._warmReadyLogged){S[k]._warmReadyLogged=true;addLog(k,'✅ MODEL WARM & READY — '+(c.message||'GPU model is loaded and ready for jobs.'),'ok');}if(!c.enabled||c.state==='off'||c.state==='error'){S[k]._warmReadyLogged=false;} }
async function pollWarm(){try{const d=await (await fetch('/api/warm')).json();for(const k of ['cb','w','u'])renderWarm(k,d[warmTools[k]]);}catch{}}
async function toggleWarm(k){const tool=warmTools[k];const d=await (await fetch('/api/warm')).json();const action=d[tool].enabled?'off':'on';const r=await fetch('/api/warm/'+tool+'/'+action,{method:'POST'});const x=await r.json();if(!r.ok){setStatus(k,'❌ '+(x.error||'Warm mode failed.'),false,true);return;}setStatus(k,action==='on'?'Loading model into GPU...':'Releasing model from GPU...');pollWarm();}
['cb','w','u'].forEach(k=>$(warmUi[k][0]).onclick=()=>toggleWarm(k));
async function health(){try{const d=await (await fetch('/api/health')).json();const all=Object.values(d.runners).every(x=>x.ready);$('runtimeText').textContent=all?'All 3 runners ready':'One or more runners missing';for(const [key,cfg] of Object.entries(d.runners)){const id=key==='chatterbox'?'cbBadge':key==='whisper'?'wBadge':'uBadge';$(id).textContent=cfg.ready?'READY':'MISSING';$(id).className='badge'+(cfg.ready?' ready':'')}}catch{$('runtimeText').textContent='Server connected · health unavailable'}}
health();pollWarm();setInterval(health,5000);setInterval(pollWarm,1000);
</script>
</body>
</html>'''

# =============================================================================
# 7. ROUTES
# =============================================================================

@app.get("/api/warm")
async def api_warm():
    return warm_state_snapshot()

@app.post("/api/warm/{tool}/{action}")
async def api_warm_control(tool: str, action: str):
    if tool not in TOOL_META:
        return JSONResponse({"error": "Unknown tool."}, status_code=404)
    if action not in {"on", "off"}:
        return JSONResponse({"error": "Action must be on or off."}, status_code=400)
    if tool_busy(tool):
        return JSONResponse({"error": f"Finish the current {TOOL_META[tool]['name']} job first."}, status_code=409)
    if action == "on":
        set_warm_state(tool, "loading", f"Loading {TOOL_META[tool]['name']} into GPU...", enabled=True)
        start_thread(start_warm_worker, tool)
        return {"success": True}
    start_thread(stop_warm_worker, tool)
    return {"success": True}


@app.get("/", response_class=HTMLResponse)
async def home():
    return HTML


@app.get("/api/health")
async def health():
    return {
        "runners": {
            "chatterbox": {"ready": runner_ready(CHATTERBOX_RUNNER), "path": str(CHATTERBOX_RUNNER)},
            "whisper": {"ready": runner_ready(WHISPER_RUNNER), "path": str(WHISPER_RUNNER)},
            "upscaler": {"ready": runner_ready(UPSCALER_RUNNER), "path": str(UPSCALER_RUNNER), "models": UPSCALER_MODELS.exists()},
        }
    }


@app.post("/chatterbox/start")
async def chatterbox_start(
    mode: str = Form(...),
    script: UploadFile = File(...),
    ref_audio: UploadFile | None = File(None),
    ref_zip: UploadFile | None = File(None),
    temperature: float = Form(1.0),
    pause: float = Form(0.5),
    max_chars: int = Form(200),
    warm: bool = Form(False),
):
    if not runner_ready(CHATTERBOX_RUNNER):
        return JSONResponse({"error": f"Chatterbox runner not found: {CHATTERBOX_RUNNER}"}, status_code=500)
    if tool_busy("chatterbox"):
        return JSONResponse({"error": "A Chatterbox job is already running."}, status_code=409)
    if mode not in {"single", "multi"}:
        return JSONResponse({"error": "Invalid mode."}, status_code=400)
    if not (0.1 <= temperature <= 2.0):
        return JSONResponse({"error": "Temperature must be 0.1–2.0."}, status_code=400)
    if not (0 <= pause <= 2):
        return JSONResponse({"error": "Pause must be 0–2 seconds."}, status_code=400)
    if not (50 <= max_chars <= 500):
        return JSONResponse({"error": "Max chars must be 50–500."}, status_code=400)
    if Path(script.filename or "").suffix.lower() != ".txt":
        return JSONResponse({"error": "Script must be .txt."}, status_code=400)

    job = create_job("chatterbox", safe_name(script.filename))
    job.warm = bool(warm and warm_enabled("chatterbox"))
    root = JOB_DIR / "chatterbox" / job.id
    root.mkdir(parents=True, exist_ok=True)
    script_path = root / "script.txt"
    await save_upload(script, script_path)
    job.output_path = root / f"{job.id}_output.wav"

    ref_path = None
    refs_dir = None
    if mode == "single":
        if ref_audio is None or Path(ref_audio.filename or "").suffix.lower() not in CHATTERBOX_AUDIO:
            return JSONResponse({"error": "Valid reference audio is required."}, status_code=400)
        ref_path = root / safe_name(ref_audio.filename)
        await save_upload(ref_audio, ref_path)
    else:
        if ref_zip is None or Path(ref_zip.filename or "").suffix.lower() != ".zip":
            return JSONResponse({"error": "Reference ZIP is required."}, status_code=400)
        ref_path = root / safe_name(ref_zip.filename)
        refs_dir = root / "refs"
        refs_dir.mkdir(parents=True, exist_ok=True)
        await save_upload(ref_zip, ref_path)
        try:
            with zipfile.ZipFile(ref_path, "r") as z:
                if z.testzip() is not None:
                    raise RuntimeError("ZIP integrity check failed.")
        except Exception as exc:
            mark_done(job, "failed", str(exc))
            return JSONResponse({"error": str(exc)}, status_code=400)

    job.input_path = script_path
    emit(job, "status", "Files uploaded successfully.", status="queued")
    start_thread(worker_chatterbox, job, mode, script_path, ref_path, refs_dir, float(temperature), float(pause), int(max_chars))
    return {"success": True, "job_id": job.id}


@app.post("/whisper/start")
async def whisper_start(file: UploadFile = File(...), language: str = Form("auto"), warm: bool = Form(False)):
    if not runner_ready(WHISPER_RUNNER):
        return JSONResponse({"error": f"Whisper runner not found: {WHISPER_RUNNER}"}, status_code=500)
    if tool_busy("whisper"):
        return JSONResponse({"error": "A Whisper job is already running."}, status_code=409)
    filename = safe_name(file.filename)
    if Path(filename).suffix.lower() not in WHISPER_AUDIO:
        return JSONResponse({"error": "Unsupported audio format."}, status_code=400)

    job = create_job("whisper", filename)
    job.warm = bool(warm and warm_enabled("whisper"))
    root = JOB_DIR / "whisper" / job.id
    root.mkdir(parents=True, exist_ok=True)
    job.input_path = UPLOAD_DIR / f"{job.id}_{filename}"
    stem = Path(filename).stem
    job.words_path = root / f"{stem}_words.json"
    job.segments_path = root / f"{stem}_segments.json"
    job.segments_txt_path = root / f"{stem}_segments.txt"
    job.words_txt_path = root / f"{stem}_words.txt"
    job.segment_srt_path = root / f"{stem}.srt"
    job.word_srt_path = root / f"{stem}_words.srt"
    await save_upload(file, job.input_path)
    emit(job, "status", "Audio uploaded successfully.", status="queued")
    start_thread(worker_whisper, job, language)
    return {"success": True, "job_id": job.id}


@app.post("/upscaler/start")
async def upscaler_start(file: UploadFile = File(...), scale: float = Form(4.0), warm: bool = Form(False)):
    if not runner_ready(UPSCALER_RUNNER):
        return JSONResponse({"error": f"Real-ESRGAN runner not found: {UPSCALER_RUNNER}"}, status_code=500)
    if tool_busy("upscaler"):
        return JSONResponse({"error": "A Real-ESRGAN job is already running."}, status_code=409)
    scale = float(scale)
    if scale not in SCALES:
        return JSONResponse({"error": "Scale must be 2, 3.5 or 4."}, status_code=400)
    filename = safe_name(file.filename)
    ext = Path(filename).suffix.lower()
    if ext not in IMAGES and ext != ".zip":
        return JSONResponse({"error": "Unsupported image/ZIP format."}, status_code=400)

    job = create_job("upscaler", filename)
    job.warm = bool(warm and warm_enabled("upscaler"))
    root = JOB_DIR / "upscaler" / job.id
    root.mkdir(parents=True, exist_ok=True)
    job.input_path = UPLOAD_DIR / f"{job.id}_{filename}"
    await save_upload(file, job.input_path)
    job.output_path = root / (f"{Path(filename).stem}_upscaled.zip" if ext == ".zip" else f"{Path(filename).stem}_upscaled.png")
    if ext == ".zip":
        try:
            with zipfile.ZipFile(job.input_path, "r") as z:
                if z.testzip() is not None:
                    raise RuntimeError("ZIP integrity check failed.")
        except Exception as exc:
            mark_done(job, "failed", str(exc))
            return JSONResponse({"error": str(exc)}, status_code=400)
    emit(job, "status", "Input uploaded successfully.", status="queued")
    start_thread(worker_upscaler, job, scale)
    return {"success": True, "job_id": job.id}


@app.websocket("/ws/{job_id}")
async def ws_events(ws: WebSocket, job_id: str):
    await ws.accept()
    with jobs_lock:
        job = jobs.get(job_id)
    if not job:
        await ws.send_json({"type": "status", "message": "Job not found.", "status": "failed"})
        await ws.close()
        return
    index = 0
    try:
        while True:
            with job.lock:
                events = job.events[index:]
                index = len(job.events)
            for event in events:
                await ws.send_json(event)
            if job.status in {"completed", "failed"}:
                await asyncio.sleep(.15)
                with job.lock:
                    events = job.events[index:]
                    index = len(job.events)
                for event in events:
                    await ws.send_json(event)
                break
            await asyncio.sleep(.1)
    except Exception:
        pass


@app.post("/cancel/{job_id}")
async def cancel(job_id: str):
    with jobs_lock:
        job = jobs.get(job_id)
    if not job:
        return JSONResponse({"error": "Job not found."}, status_code=404)
    if job.warm:
        tool = job.tool
        with WARM_LOCKS[tool]:
            cur = WARM_CURRENT.get(tool)
            if cur and cur["job"].id == job.id:
                cur["error"] = "Job cancelled."
                cur["event"].set()
        was_enabled = WARM_STATE[tool]["enabled"]
        stop_warm_worker(tool)
        if was_enabled:
            set_warm_state(tool, "loading", "Restarting warm worker...", enabled=True)
            start_thread(start_warm_worker, tool)
    elif job.process and job.process.poll() is None:
        try:
            job.process.terminate()
            try:
                job.process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                job.process.kill()
        except Exception as exc:
            return JSONResponse({"error": str(exc)}, status_code=500)
    mark_done(job, "failed", "Job cancelled.")
    return {"success": True}

# =============================================================================
# 8. CLOUDFLARE + SERVER
# =============================================================================

def free_port():
    for port in range(8000, 8100):
        s = socket.socket()
        try:
            s.bind(("127.0.0.1", port))
            return port
        except OSError:
            pass
        finally:
            s.close()
    raise RuntimeError("No free port found.")


if not CLOUDFLARED.exists():
    print("⬇️ Installing cloudflared…", flush=True)
    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", str(CLOUDFLARED),
        ],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

PORT = free_port()
server_thread = threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning"), daemon=True)
server_thread.start()
time.sleep(1.5)

print("\n" + "=" * 86)
print("🚀 AI FLOWS HUB")
print("=" * 86)
print(f"Base       : {BASE_DIR}")
print(f"Chatterbox : {'READY' if runner_ready(CHATTERBOX_RUNNER) else 'MISSING'}")
print(f"Whisper    : {'READY' if runner_ready(WHISPER_RUNNER) else 'MISSING'}")
print(f"Real-ESRGAN: {'READY' if runner_ready(UPSCALER_RUNNER) else 'MISSING'}")
print(f"Port       : {PORT}")
print(f"Drive      : MyDrive/{DRIVE_FOLDER_NAME}")
print("\n🌐 Starting Cloudflare HTTPS tunnel…")

tunnel = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
start = time.time()
while time.time() - start < 40:
    line = tunnel.stdout.readline() if tunnel.stdout else ""
    if not line:
        continue
    line = line.strip()
    print(line, flush=True)
    m = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

print("\n" + "=" * 86)
if public_url:
    print("✅ AI FLOWS HUB READY")
    print(f"🌐 OPEN: {public_url}")
else:
    print("⚠️ Tunnel URL not detected automatically.")
    print(f"FastAPI: http://127.0.0.1:{PORT}")
print("Tools: Chatterbox · Whisper · Real-ESRGAN")
print("=" * 86)

✅ Google Drive ready: MyDrive/AI Flows Hub Downloads
✅ Whisper runner is up-to-date (v2.0.0): /content/run_whisper.py
✅ Verified Whisper runner CLI arguments successfully.

🚀 AI FLOWS HUB
Base       : /content
Chatterbox : READY
Whisper    : READY
Real-ESRGAN: READY
Port       : 8002
Drive      : MyDrive/AI Flows Hub Downloads

🌐 Starting Cloudflare HTTPS tunnel…
2026-08-27T07:11:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-27T07:11:47Z INF Request